> **Attribution:** Adapted from [Keithland89/StudioLens-for-Copilot-Studio](https://github.com/Keithland89/StudioLens-for-Copilot-Studio), licensed under the MIT License.
>
> Adapted by the ESS Insights team for ESS-only scoping.

# Copilot Agent Transcript — Direct Ingester + Parser (Fabric)

**Ingests** Copilot Studio conversation transcripts **directly from Dataverse** (the `conversationtranscripts` table, via the Web API with an app registration) **and parses** them into the flat Delta fact tables the **AI Business Value Dashboard — 1905 Extra** PBIP consumes. No manual CSV export step.

Two source modes (set `SOURCE_MODE` in the CONFIG cell):

| `SOURCE_MODE` | Transcript source | Use when |
|---|---|---|
| `'dataverse'` *(default)* | Live Web API pull from `conversationtranscripts`, paged + incremental | Hands-off, scheduled pipeline runs |
| `'files'` | Reads `conversationtranscripts.csv` already in `Files/` | Air-gapped / one-off / no app reg |

The OCV **`feedback`** export remains file-based (it lives in the M365 admin centre, not Dataverse) — drop it in `Files/` or skip it; the notebook tolerates its absence.

```
Dataverse  conversationtranscripts  ──(Web API, app-reg auth, paged)──┐
                                                                       ▼
                          this notebook  (ingest → raw Delta → parse JSON content)
                                                                       ▼
   agent_sessions · agent_turns · agent_errors · agent_subagents
   agent_catalogue · agent_performance · user_feedback · user_feedback_quarantine
                                                                       ▼
                                 PBIP (Lakehouse SQL endpoint / Import)
```

**Output tables** (consumed by the model's *Agent Sessions, Agent Turns, Agent Errors, Agent Sub-Agent Calls, Agent Catalogue,* and *Agent Performance* tables):

| Delta table | Grain |
|---|---|
| `conversationtranscripts_raw` | raw landing — one row per transcript (optional, set `RAW_TABLE=''` to skip) |
| `agent_sessions` | one row per environment-aware conversation |
| `agent_turns` | one row per user/bot message |
| `agent_errors` | one row per `ErrorTraceData` event |
| `agent_subagents` | one row per `ConnectedAgent` invocation |
| `agent_catalogue` | conformed agent dimension (one row per `canonical_agent_key`) |
| `agent_performance` | per-conversation KPI fact |
| `user_feedback` | registry-mapped transcript and external feedback |
| `user_feedback_quarantine` | identifiers only for unmapped external feedback |

**Permissions (`SOURCE_MODE='dataverse'`)** — an Entra **app registration** added as an **Application User** in the target Dataverse environment, with a security role granting **Read** on the **Conversation Transcript** table (e.g. *System Administrator*, *System Customizer*, or a custom least-privilege role). The notebook authenticates client-credentials against `{DATAVERSE_URL}/.default` — no Microsoft Graph permissions are required for transcripts. (`'files'` mode needs no app reg at all.)

**Pipeline wiring** — you can schedule this notebook directly or add it to your own Fabric pipeline. Tag the **CONFIG cell as `parameters`** (cell ⋯ menu → *Toggle parameter cell*) so your orchestration can override `LOOKBACK_DAYS`, `WRITE_MODE`, `SOURCE_MODE`, etc. `WRITE_MODE='merge'` is the safe default for bounded incremental reruns; full snapshots use `LOOKBACK_DAYS=0` with `'overwrite'`. The registry CSV is required, and `AGENT_SCOPE_MODE` defaults to fail-closed `ESS Safe`.


## 1. Configuration  ·  *(mark this as the pipeline `parameters` cell)*

Set `SOURCE_MODE`, credentials, `AGENT_REGISTRY_FILE`, and the exact `AGENT_SCOPE_MODE` (`ESS Safe`, `Selected Agents`, or `All Agents`). Leave `OUTPUT_PREFIX=''` for a standard Lakehouse; use `dbo` only when Lakehouse schemas are enabled.

For a scheduled run, toggle this cell as a parameter cell. In production, replace `CLIENT_SECRET` with `notebookutils.credentials.getSecret(...)`.


In [ ]:
# === CONFIG ===  (tag this cell as the pipeline `parameters` cell)

# --- Source mode ---
SOURCE_MODE   = 'dataverse'   # 'dataverse' = live Web API pull; 'files' = read CSV already in Files/

# --- Dataverse environment(s) — used when SOURCE_MODE='dataverse' ---
# ONE environment:   set DATAVERSE_URL and leave DATAVERSE_URLS = [].
# MANY environments: set DATAVERSE_URLS to a list of env URLs (DATAVERSE_URL is then ignored).
#   The app registration must be added as an Application User in EVERY listed environment.
DATAVERSE_URL  = 'https://orgXXXXXXXX.crm.dynamics.com'   # single environment URL, NO trailing slash
DATAVERSE_URLS = [
    # 'https://org1.crm.dynamics.com',
    # 'https://org2.crm.dynamics.com',
]
TAG_ENVIRONMENT = True   # add a 'SourceEnvironment' column to every row (lets the PBIP slice by env)

TENANT_ID     = '<your-tenant-guid>'
CLIENT_ID     = '<your-app-reg-client-id>'
CLIENT_SECRET = '<your-client-secret-value>'   # prod: notebookutils.credentials.getSecret('<kv-uri>', '<name>')
LOOKBACK_DAYS = 7        # incremental window on conversationstarttime; 0 = full pull (all rows)
PAGE_SIZE     = 5000     # Dataverse OData max page size (server caps at 5000)
RAW_TABLE     = 'conversationtranscripts_raw'   # raw landing Delta table. The PBIT does NOT
                                                    # need it (agent_* tables are derived below), so on
                                                    # VERY large tenants set RAW_TABLE = '' to skip the
                                                    # content-heavy landing entirely and save time/space.

# Resolve the environment list: DATAVERSE_URLS wins if non-empty, else fall back to the single URL.
# Placeholder/empty entries are dropped so an unset DATAVERSE_URL doesn't get pulled by accident.
DATAVERSE_ENVIRONMENTS = [
    u.rstrip('/') for u in (DATAVERSE_URLS if DATAVERSE_URLS else [DATAVERSE_URL])
    if u and not u.startswith('<') and 'XXXX' not in u
]

# --- Files (fallback transcripts) ---
SOURCE_DIR        = 'Files/copilot_transcripts'
TRANSCRIPTS_FILE  = f'{SOURCE_DIR}/conversationtranscripts.csv'   # used only when SOURCE_MODE='files'
TOPIC_OVERRIDES_FILE = '/lakehouse/default/Files/config/customer-topic-overrides.csv'  # optional output from tools/ess_topic_tuner.py
FEEDBACK_EVENTS_FILE = '/lakehouse/default/Files/feedback/feedback-events.csv'  # optional output from tools/ess_feedback_normalizer.py

# --- Output ---
OUTPUT_PREFIX = ''           # Spark table prefix. Leave blank for standard Lakehouses; use 'dbo' only when schemas are enabled.
WRITE_MODE    = 'merge'      # safe default for scheduled incremental pulls; use overwrite only with a full Dataverse pull
TEXT_TRUNCATE = 500          # max chars kept for free-text fields (PII-safe preview)


# __INCREMENTAL_OVERWRITE_GUARD__
if SOURCE_MODE == 'dataverse' and LOOKBACK_DAYS > 0 and WRITE_MODE == 'overwrite':
    raise ValueError(
        "Unsafe configuration: overwrite with a bounded Dataverse lookback would delete older history. "
        "Use WRITE_MODE='merge', or set LOOKBACK_DAYS=0 for an intentional full snapshot.")

# --- Large-tenant tuning ---
PANDAS_TO_SPARK_CHUNK_ROWS = 20000   # rows per chunk for the slim agent_* tables (pandas -> Spark Delta)
RAW_TABLE_CHUNK_ROWS       = 2000    # SMALLER chunk for the raw landing: each row carries the full
                                     # conversation `content` JSON (tens of KB), so a 20k-row task can
                                     # exceed spark.rpc.message.maxSize. 2k keeps each task well under it.

# --- Pull resilience & parallelism (large / customer tenant pulls) ---
REQUEST_TIMEOUT    = 120     # per-page HTTP timeout (seconds) for each Dataverse GET
MAX_RETRIES        = 5       # transient-failure retries per request (429 / 5xx / timeout / conn-reset)
RETRY_BACKOFF      = 2.0     # exponential backoff base: sleep ~= RETRY_BACKOFF**attempt (+ jitter); honours Retry-After
MAX_PARALLEL_PULLS = 4       # concurrent fetch workers across (environment x date-window). 1 = fully serial.
PULL_WINDOWS       = 4       # split the LOOKBACK_DAYS range into this many date-windows pulled IN PARALLEL.
                             # Only applies when LOOKBACK_DAYS > 0. A full pull (LOOKBACK_DAYS=0) streams
                             # serially per environment (still retry-hardened, just one window).

# === AGENT REGISTRY CONFIG (generated) ===
AGENT_REGISTRY_FILE = '/lakehouse/default/Files/config/agent-registry.csv'
AGENT_SCOPE_MODE = 'ESS Safe'
_AGENT_SCOPE_MODES = {'ESS Safe', 'Selected Agents', 'All Agents'}
if AGENT_SCOPE_MODE not in _AGENT_SCOPE_MODES:
    raise ValueError(
        'AGENT_SCOPE_MODE must be exactly one of: ESS Safe, Selected Agents, All Agents')
# === END AGENT REGISTRY CONFIG ===


## 1b. Preflight  ·  *(run this FIRST in `dataverse` mode — fails fast with a clear verdict)*

A 5-second check before the full pull: gets a client-credentials **token**, then does a single `$top=1` GET on `conversationtranscripts`. It tells you exactly which of the three things is wrong (credentials / Application User permission / empty table) instead of dying mid-pagination. No-op in `files` mode.


In [ ]:
# === 1b. PREFLIGHT (Dataverse auth + permission check) ===
# Run this on its own BEFORE the full ingest. Checks EVERY environment in DATAVERSE_ENVIRONMENTS.
# Skips silently in 'files' mode.
if SOURCE_MODE != 'dataverse':
    print("SOURCE_MODE='files' — preflight skipped (no Dataverse auth needed).")
elif not DATAVERSE_ENVIRONMENTS:
    raise SystemExit("✗ No valid environments. Set DATAVERSE_URL or DATAVERSE_URLS in CONFIG "
                     "(placeholder/<...>/XXXX values are ignored).")
else:
    import requests

    def _preflight_one(base):
        """Returns (ok: bool, message: str) for a single environment URL."""
        base = base.rstrip('/')
        # 1) Token (scope is per-environment)
        try:
            t = requests.post(
                f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token',
                data={'grant_type': 'client_credentials', 'client_id': CLIENT_ID,
                      'client_secret': CLIENT_SECRET, 'scope': f'{base}/.default'},
                timeout=60)
        except Exception as e:
            return False, f'token request failed to send: {e}'
        if t.status_code != 200:
            err = t.json().get('error', '?') if t.headers.get('content-type','').startswith('application/json') else t.text[:200]
            return False, f"token HTTP {t.status_code}: {err}  (check TENANT_ID/CLIENT_ID/CLIENT_SECRET; 'invalid_client' = bad secret)"
        tok = t.json()['access_token']
        # 2) Single-row GET (tests Application User + table read in THIS environment)
        try:
            r = requests.get(
                f'{base}/api/data/v9.2/conversationtranscripts?$select=conversationtranscriptid&$top=1',
                headers={'Authorization': f'Bearer {tok}', 'Accept': 'application/json',
                         'OData-MaxVersion': '4.0', 'OData-Version': '4.0'}, timeout=60)
        except Exception as e:
            return False, f'data GET failed to send: {e}'
        if r.status_code == 403:
            return False, ("403 Forbidden — token valid but app has no access HERE. Add it as an "
                           "Application User in THIS environment (Power Platform Admin Center → "
                           "Settings → Users + permissions → Application users) with READ on conversationtranscripts.")
        if r.status_code == 404:
            return False, "404 — wrong URL (must be https://<org>.crm.dynamics.com, no trailing slash)."
        if r.status_code != 200:
            return False, f'HTTP {r.status_code}: {r.text[:200]}'
        n = len(r.json().get('value', []))
        return True, ('read OK' + ('' if n else ' — but 0 rows on $top=1 (env may have no transcripts yet)'))

    print(f'Preflight across {len(DATAVERSE_ENVIRONMENTS)} environment(s):\n')
    _results = []
    for _env in DATAVERSE_ENVIRONMENTS:
        ok, msg = _preflight_one(_env)
        _results.append(ok)
        print(f"  {'✓' if ok else '✗'} {_env}\n      {msg}")

    _passed = sum(_results)
    print(f'\n{_passed}/{len(DATAVERSE_ENVIRONMENTS)} passed.')
    if _passed < len(DATAVERSE_ENVIRONMENTS):
        raise SystemExit(
            '✗ PREFLIGHT FAILED — every configured environment must pass before ingestion. '
            'No Delta table was changed; fix the messages above and retry.'
        )
    print('✓ PREFLIGHT PASSED — every configured environment is readable; safe to run cell 2a.')


## 2. Ingest & load the source data

**2a — Ingest from Dataverse** (runs when `SOURCE_MODE='dataverse'`): **loops over every environment** in `DATAVERSE_ENVIRONMENTS`, authenticates the app registration (token scope is per-environment), pages through `conversationtranscripts` with the OData Web API (incremental filter on `conversationstarttime` when `LOOKBACK_DAYS > 0`), tags each row with `SourceEnvironment` (when `TAG_ENVIRONMENT=True`), concatenates all environments into one frame `tx`, and optionally lands the raw rows to `RAW_TABLE`. Any environment, token, or window failure aborts before a Delta write so an incomplete pull cannot replace prior data. In `'files'` mode this cell is a no-op and `tx` is read from CSV in 2b.

**2b — Parse JSON content**: locate the `content` column, parse each transcript's JSON, and keep the rows that parse — producing the `parsed` frame the rest of the notebook consumes unchanged. `SourceEnvironment` (if added) rides along through the parse. Transcript volumes are conversation-level (thousands–tens-of-thousands of rows), so a driver-side parse is fine; for very large tenants switch the parse loop to `df.rdd.mapPartitions(...)`.


In [ ]:
# === 2a. INGEST FROM DATAVERSE (Web API) — parallel windows + retry/backoff ===
# Loops over every environment in DATAVERSE_ENVIRONMENTS. For an incremental pull (LOOKBACK_DAYS > 0)
# each environment's date range is split into PULL_WINDOWS half-open slices fetched CONCURRENTLY (up to
# MAX_PARALLEL_PULLS workers across environment x window). Every page GET is wrapped in exponential
# backoff/retry (429 / 5xx / timeout / conn-reset, honouring Retry-After) so a large pull won't crash on
# a single transient timeout. Rows are tagged with SourceEnvironment (when TAG_ENVIRONMENT) and all
# slices are concatenated (de-duped on conversationtranscriptid) into one frame `tx`.
import requests, time, random
import pandas as pd
from datetime import datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

_SELECT = ('conversationtranscriptid,content,conversationstarttime,schemaversion,'
           'schematype,name,metadata,statecode,statuscode,createdon')
# AGENT IDENTITY: the bot lookup carries the real Copilot Studio agent (schemaname). Expand it so
# single-agent transcripts (no ConnectedAgentInitializeTraceData in content) still resolve a primary
# agent. Falls back to no-expand per slice if the nav property is unavailable in that environment.
_EXPAND = 'bot_conversationtranscriptid($select=schemaname,name,botid)'
_RETRYABLE_STATUS = {429, 500, 502, 503, 504}

def _dataverse_token(base):
    """Client-credentials token for one environment ({base}/.default)."""
    resp = requests.post(
        f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token',
        data={'grant_type': 'client_credentials', 'client_id': CLIENT_ID,
              'client_secret': CLIENT_SECRET, 'scope': f'{base.rstrip("/")}/.default'},
        timeout=60)
    resp.raise_for_status()
    return resp.json()['access_token']

def _get_with_retry(url, headers):
    """GET one page with exponential backoff on transient failures (timeout / conn-reset / 429 / 5xx).
    Honours Retry-After when present. Returns the response for success AND non-retryable HTTP codes
    (e.g. 400/401/403/404) so the caller can inspect them; raises only after MAX_RETRIES."""
    _max     = int(globals().get('MAX_RETRIES', 5))
    _backoff = float(globals().get('RETRY_BACKOFF', 2.0))
    _timeout = int(globals().get('REQUEST_TIMEOUT', 120))
    last = None
    for attempt in range(_max + 1):
        wait = None
        try:
            r = requests.get(url, headers=headers, timeout=_timeout)
            if r.status_code not in _RETRYABLE_STATUS:
                return r
            last = requests.HTTPError(f'HTTP {r.status_code}', response=r)
            ra = r.headers.get('Retry-After')
            if ra:
                try:
                    wait = float(ra)
                except ValueError:
                    wait = None
        except (requests.Timeout, requests.ConnectionError) as e:
            last = e
        if attempt >= _max:
            break
        if wait is None:
            wait = (_backoff ** attempt) + random.uniform(0, 0.75)
        time.sleep(min(wait, 120))
    if last is None:
        raise RuntimeError('request failed after retries')
    raise last

def _fetch_window(base, headers, win_start, win_end):
    """Serial nextLink paging for ONE (environment, [win_start, win_end)) date slice.
    win_start / win_end are ISO-Z strings or None (None = unbounded on that side)."""
    base = base.rstrip('/')
    flt = []
    if win_start:
        flt.append(f'conversationstarttime ge {win_start}')
    if win_end:
        flt.append(f'conversationstarttime lt {win_end}')
    filter_q = ('&$filter=' + ' and '.join(flt)) if flt else ''
    expand = True
    url = (f'{base}/api/data/v9.2/conversationtranscripts?$select={_SELECT}'
           f'&$expand={_EXPAND}{filter_q}')
    rows = []
    while url:
        r = _get_with_retry(url, headers)
        if r.status_code == 400 and expand:
            # bot lookup nav property not exposed in this env -> retry this slice without expand,
            # PRESERVING the date filter (the old code dropped the filter too).
            expand = False
            url = f'{base}/api/data/v9.2/conversationtranscripts?$select={_SELECT}{filter_q}'
            continue
        r.raise_for_status()
        body = r.json()
        rows.extend(body.get('value', []))
        url = body.get('@odata.nextLink')
    return rows

def _iter_windows():
    """Date slices for the pull. Full pull (LOOKBACK_DAYS=0) or PULL_WINDOWS<=1 -> one unbounded slice.
    Otherwise split [now-LOOKBACK_DAYS, now] into PULL_WINDOWS half-open slices (last one open-ended)."""
    n  = max(1, int(globals().get('PULL_WINDOWS', 1)))
    lb = LOOKBACK_DAYS if ('LOOKBACK_DAYS' in globals() and LOOKBACK_DAYS) else 0
    if not lb or lb <= 0 or n == 1:
        return [(None, None)]
    now   = datetime.now(timezone.utc)
    start = now - timedelta(days=int(lb))
    step  = (now - start) / n
    edges = [start + step * i for i in range(n)] + [now]
    fmt   = lambda d: d.strftime('%Y-%m-%dT%H:%M:%SZ')
    wins  = []
    for i in range(n):
        s = fmt(edges[i])
        e = None if i == n - 1 else fmt(edges[i + 1])   # last slice open-ended -> catches rows up to now
        wins.append((s, e))
    return wins

tx = None
if SOURCE_MODE == 'dataverse':
    if not DATAVERSE_ENVIRONMENTS:
        raise SystemExit("✗ No valid environments. Set DATAVERSE_URL or DATAVERSE_URLS in CONFIG.")
    _windows = _iter_windows()
    _workers = max(1, int(globals().get('MAX_PARALLEL_PULLS', 4)))
    print(f'Ingesting conversationtranscripts from {len(DATAVERSE_ENVIRONMENTS)} environment(s) '
          f'x {len(_windows)} window(s)  (LOOKBACK_DAYS={LOOKBACK_DAYS}, 0=full; '
          f'up to {_workers} parallel) ...\n')

    # Pre-fetch one token per environment SERIALLY so auth failures are clear and AAD isn't hammered.
    _tokens, _bad_envs = {}, {}
    for _env in DATAVERSE_ENVIRONMENTS:
        try:
            _tokens[_env.rstrip('/')] = _dataverse_token(_env)
        except Exception as _e:
            _bad_envs[_env] = _e
            print(f'  ⚠ SKIPPED {_env} — token failed: {_e}  (run the preflight to diagnose)')

    def _headers_for(_env):
        return {
            'Authorization':   f'Bearer {_tokens[_env]}',
            'Accept':          'application/json',
            'OData-MaxVersion':'4.0',
            'OData-Version':   '4.0',
            'Prefer':          f'odata.maxpagesize={PAGE_SIZE}',
        }

    _tasks = [(_env, ws, we) for _env in _tokens for (ws, we) in _windows]

    def _run(task):
        _e, ws, we = task
        return _e, (ws, we), _fetch_window(_e, _headers_for(_e), ws, we)

    _env_rows = {_env: [] for _env in _tokens}
    _errs = []
    _n = max(1, min(_workers, len(_tasks) or 1))
    with ThreadPoolExecutor(max_workers=_n) as _ex:
        _futs = {_ex.submit(_run, _t): _t for _t in _tasks}
        for _fut in as_completed(_futs):
            _t = _futs[_fut]
            try:
                _e2, _win, _rows = _fut.result()
                _env_rows[_e2].extend(_rows)
                print(f'    {_t[0]}  [{_t[1] or "start"} .. {_t[2] or "now"}]: +{len(_rows):,}')
            except Exception as _ex2:
                _errs.append((_t, _ex2))
                print(f'    ⚠ window FAILED {_t[0]} [{_t[1]} .. {_t[2]}]: {_ex2}')

    _frames = []
    for _env, _rows in _env_rows.items():
        if not _rows:
            print(f'    {_env}: 0 rows'); continue
        _df = pd.DataFrame(_rows).fillna('')
        _df = _df.rename(columns={col: col.lower() for col in _df.columns})
        # Drop OData annotation columns (e.g. '@odata.etag') that aren't real fields.
        _df = _df[[col for col in _df.columns if not col.startswith('@') and not col.startswith('_')]]
        # Half-open windows don't overlap, but de-dup defensively on the natural key.
        if 'conversationtranscriptid' in _df.columns:
            _df = _df.drop_duplicates(subset=['conversationtranscriptid'])
        if TAG_ENVIRONMENT:
            _df['SourceEnvironment'] = _env
        print(f'    {_env}: {len(_df):,} rows total')
        _frames.append(_df)

    if _bad_envs or _errs:
        raise SystemExit(
            f'✗ INGEST ABORTED — token failures={len(_bad_envs)}, window failures={len(_errs)}. '
            'No Delta table was changed; fix the failures and retry the complete configured set.'
        )

    tx = pd.concat(_frames, ignore_index=True) if _frames else pd.DataFrame()
    print(f'\nTotal conversationtranscripts pulled across environments: {len(tx):,}  '
          f'(windows failed: {len(_errs)}, envs skipped: {len(_bad_envs)})')
    if not len(tx):
        raise SystemExit('✗ INGEST ABORTED — the configured environments returned 0 transcript rows. No Delta table was changed.')
    if RAW_TABLE and len(tx) and WRITE_MODE != 'merge':
        # Avoid giant Spark task serialization from one huge pandas->Spark conversion.
        # The raw `content` JSON is heavy, so use the SMALLER raw chunk size: a 20k-row task of
        # full-conversation JSON can exceed spark.rpc.message.maxSize (the customer's 469MB error).
        _chunk_rows = int(globals().get('RAW_TABLE_CHUNK_ROWS', 2000))
        if WRITE_MODE == 'overwrite':
            spark.sql(f'DROP TABLE IF EXISTS {RAW_TABLE}')
        _raw_pdf = tx.astype(str).where(pd.notna(tx), None)
        _raw_schema = StructType([StructField(c, StringType(), True) for c in _raw_pdf.columns])
        _chunk_count = 0
        for _i in range(0, len(_raw_pdf), _chunk_rows):
            _part = _raw_pdf.iloc[_i:_i + _chunk_rows]
            _mode = 'append' if (WRITE_MODE == 'overwrite' or _chunk_count > 0) else WRITE_MODE
            (spark.createDataFrame(_part, schema=_raw_schema)
                .write.mode(_mode)
                .option('overwriteSchema', 'true')
                .option('delta.columnMapping.mode', 'name')
                .option('delta.minReaderVersion', '2')
                .option('delta.minWriterVersion', '5')
                .format('delta').saveAsTable(RAW_TABLE))
            _chunk_count += 1
        print(f'  raw landed -> {RAW_TABLE} ({WRITE_MODE}, chunks={_chunk_count}, chunk_rows={_chunk_rows})')
    elif RAW_TABLE and WRITE_MODE == 'merge':
        print(f'  raw landing skipped for WRITE_MODE=merge; curated Delta tables are merged safely below.')
else:
    print("SOURCE_MODE='files' — transcripts will be read from CSV in cell 2b.")

In [ ]:
# === 2b. PARSE JSON CONTENT ===
import json, re
from datetime import datetime, timezone
import pandas as pd

def _read_csv(path):
    return (spark.read
            .option('header', True)
            .option('multiLine', True)
            .option('escape', '"')
            .option('encoding', 'UTF-8')
            .csv(path))

# `tx` is populated by cell 2a when SOURCE_MODE='dataverse'.
# In 'files' mode (or if 2a was skipped) read the CSV here instead.
if SOURCE_MODE == 'files' or 'tx' not in dir() or tx is None:
    transcripts_sdf = _read_csv(TRANSCRIPTS_FILE)
    transcripts_sdf = transcripts_sdf.toDF(*[c.lower() for c in transcripts_sdf.columns])
    tx = transcripts_sdf.toPandas().fillna('')

print(f'conversationtranscripts rows: {len(tx):,}')
if len(tx) == 0:
    raise SystemExit('✗ INGEST ABORTED — the source contains 0 transcript rows. No Delta table was changed.')

content_col = next((c for c in tx.columns if c.lower().endswith('content')), None)
if content_col is None and len(tx) == 0:
    print('  conversationtranscripts returned 0 rows -> writing empty agent tables (schema preserved).')
elif content_col is None:
    raise RuntimeError('No content column found in conversationtranscripts')

def _safe_json(v):
    if not v:
        return None
    try:
        return json.loads(v)
    except Exception:
        return None

if content_col is None:
    parsed = tx.iloc[0:0].copy()
    parsed['content_json'] = pd.Series([], dtype=object)
else:
    tx['content_json'] = tx[content_col].map(_safe_json)
    parsed = tx[tx['content_json'].notna()].copy()
    # Drop structurally-corrupt rows. A malformed CSV export (embedded commas/quotes in the JSON
    # content) can shift fields so a few rows have a parseable 'content' but garbage in the other
    # columns (e.g. conversationtranscriptid = 'role":0}'). A real id is a GUID; rows whose id
    # isn't a GUID are export corruption and would otherwise fail datetime/number typing in the PBIP.
    _idcol = next((c for c in parsed.columns if c.lower() == 'conversationtranscriptid'), None)
    if _idcol is not None and len(parsed):
        _guid = re.compile(r'^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$')
        _before = len(parsed)
        parsed = parsed[parsed[_idcol].map(lambda v: bool(_guid.match(str(v).strip())))].copy()
        if _before != len(parsed):
            print(f'  dropped {_before - len(parsed):,} structurally-corrupt row(s) (non-GUID transcript id)')
print(f'rows with valid JSON content: {len(parsed):,}')
if len(parsed) == 0:
    raise SystemExit('✗ INGEST ABORTED — no rows contain valid transcript JSON. No Delta table was changed.')

# --- AGENT IDENTITY from the bot lookup (reliable even when content JSON is corrupt) ---
# The Dataverse $expand yields a nested 'bot_conversationtranscriptid' dict; the CSV export
# carries a flat 'bot_conversationtranscriptid.schemaname' column. Surface one cleaned column.
def _clean_schema(val):
    """Keep a bot schemaname only if it looks like a real schema token, not CSV-corruption
    bleed (some malformed CSV exports leak JSON/content into this column). Reject JSON chars + whitespace."""
    if val is None:
        return None
    s = str(val).strip()
    if not s or len(s) > 128:
        return None
    if any(ch in s for ch in ('{', '}', '"', ':', '\n', '\t', ' ', ',', '<', '>', '[', ']')):
        return None
    return s

def _row_schema(row):
    for c in ('bot_conversationtranscriptid.schemaname', 'bot_schemaname', 'schemaname'):
        if c in row and pd.notna(row.get(c)):
            v = _clean_schema(row.get(c))
            if v:
                return v
    nb = row.get('bot_conversationtranscriptid')
    if isinstance(nb, dict):
        return _clean_schema(nb.get('schemaname'))
    return None

if len(parsed):
    parsed['agent_schema_hint'] = parsed.apply(_row_schema, axis=1)
else:
    parsed['agent_schema_hint'] = pd.Series([], dtype=object)
print(f'rows with a bot schemaname hint: {int(parsed["agent_schema_hint"].notna().sum()) if len(parsed) else 0:,}')

# === AGENT REGISTRY RUNTIME (generated) ===
import csv as _registry_csv
import hashlib as _registry_hashlib

_REGISTRY_COLUMNS = {
    'RegistryVersion', 'AgentKey', 'AgentLabel', 'IsEss', 'Selected', 'Enabled',
    'EnvironmentKey', 'EnvironmentName', 'EnvironmentUrl', 'AliasType', 'AliasValue'}
_REGISTRY_ALIAS_TYPES = {'TranscriptId', 'TranscriptSchema', 'CreditId', 'M365Title'}
_IDENTITY_COLUMNS = [
    'conversation_natural_key', 'canonical_agent_key', 'canonical_agent_name',
    'canonical_environment_key', 'agent_mapping_status', 'raw_agent_id',
    'raw_agent_schema', 'raw_agent_name', 'raw_environment', 'source_environment']

def _registry_bool(value, field):
    normalized = str(value or '').strip().casefold()
    if normalized in {'true', '1', 'yes'}:
        return True
    if normalized in {'false', '0', 'no'}:
        return False
    raise ValueError(f'{field} must be true or false, not {value!r}')

def normalize_agent_alias(alias_type, value):
    normalized = str(value or '').strip()
    if alias_type in {'TranscriptId', 'CreditId'}:
        normalized = normalized.strip('{}')
        if alias_type == 'CreditId' and normalized.casefold().startswith('p_'):
            normalized = normalized[2:]
    return normalized.casefold()

def load_agent_registry(path):
    try:
        with open(path, 'r', encoding='utf-8-sig', newline='') as handle:
            reader = _registry_csv.DictReader(handle)
            if not reader.fieldnames:
                raise ValueError(f'Agent registry has no header: {path}')
            missing = sorted(_REGISTRY_COLUMNS - set(reader.fieldnames))
            if missing:
                raise ValueError(f'Agent registry is missing columns: {missing}')
            source_rows = list(reader)
    except FileNotFoundError as error:
        raise ValueError(f'Agent registry not found: {path}') from error

    rows, metadata_by_key, aliases = [], {}, {}
    for row_number, row in enumerate(source_rows, start=2):
        version = str(row.get('RegistryVersion') or '').strip()
        agent_key = str(row.get('AgentKey') or '').strip()
        agent_label = str(row.get('AgentLabel') or '').strip()
        alias_type = str(row.get('AliasType') or '').strip()
        alias_value = str(row.get('AliasValue') or '').strip()
        environment_key = str(row.get('EnvironmentKey') or '').strip()
        if version != '1':
            raise ValueError(
                f'Agent registry row {row_number} has unsupported version {version!r}')
        if not agent_key or not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9._:-]*', agent_key):
            raise ValueError(f'Agent registry row {row_number} has invalid AgentKey')
        if not agent_label:
            raise ValueError(f'Agent registry row {row_number} has no AgentLabel')
        if alias_type not in _REGISTRY_ALIAS_TYPES:
            raise ValueError(f'Agent registry row {row_number} has invalid AliasType')
        alias_normalized = normalize_agent_alias(alias_type, alias_value)
        if not alias_normalized:
            raise ValueError(f'Agent registry row {row_number} has no AliasValue')
        normalized = {
            'RegistryVersion': '1',
            'AgentKey': agent_key,
            'AgentLabel': agent_label,
            'IsEss': _registry_bool(row.get('IsEss'), 'IsEss'),
            'Selected': _registry_bool(row.get('Selected'), 'Selected'),
            'Enabled': _registry_bool(row.get('Enabled'), 'Enabled'),
            'EnvironmentKey': environment_key,
            'EnvironmentName': str(row.get('EnvironmentName') or '').strip(),
            'EnvironmentUrl': str(row.get('EnvironmentUrl') or '').strip(),
            'AliasType': alias_type,
            'AliasValue': alias_value,
            'AliasValueNormalized': alias_normalized,
        }
        metadata = (
            normalized['AgentLabel'], normalized['IsEss'],
            normalized['Selected'], normalized['Enabled'])
        old_metadata = metadata_by_key.setdefault(agent_key.casefold(), metadata)
        if old_metadata != metadata:
            raise ValueError(
                f'Agent registry has conflicting metadata for AgentKey {agent_key!r}')
        alias_key = (
            environment_key.casefold(), alias_type.casefold(), alias_normalized)
        old_key = aliases.setdefault(alias_key, agent_key)
        if old_key.casefold() != agent_key.casefold():
            raise ValueError(
                'Agent registry alias maps to multiple agents: '
                f'{environment_key or "*"} / {alias_type} / {alias_value}')
        rows.append(normalized)
    if not rows:
        raise ValueError('Agent registry has no rows')
    return rows

def active_agent_keys(rows, scope_mode):
    if scope_mode not in _AGENT_SCOPE_MODES:
        raise ValueError(
            'AGENT_SCOPE_MODE must be exactly one of: ESS Safe, Selected Agents, All Agents')
    keys = {
        row['AgentKey'] for row in rows
        if row['Enabled']
        and (scope_mode != 'ESS Safe' or row['IsEss'])
        and (scope_mode != 'Selected Agents' or row['Selected'])
    }
    if scope_mode != 'All Agents' and not keys:
        raise ValueError(
            f'Agent registry has no enabled agents for scope mode {scope_mode!r}')
    return keys

def registry_environment_key(rows, raw_environment):
    raw = str(raw_environment or '').strip().rstrip('/').casefold()
    if not raw:
        return ''
    keys = {
        row['EnvironmentKey'] for row in rows
        if row['Enabled'] and any(
            raw == str(row[field] or '').strip().rstrip('/').casefold()
            for field in ('EnvironmentKey', 'EnvironmentName', 'EnvironmentUrl'))
    }
    if len(keys) > 1:
        raise ValueError(f'Agent registry environment is ambiguous: {raw_environment!r}')
    return next(iter(keys), '')

def resolve_registry_alias(rows, alias_type, alias_value, active_keys=None, environment_key=''):
    normalized = normalize_agent_alias(alias_type, alias_value)
    if not normalized:
        return None
    candidates = [
        row for row in rows
        if row['Enabled']
        and row['AliasType'] == alias_type
        and row['AliasValueNormalized'] == normalized
        and (active_keys is None or row['AgentKey'] in active_keys)
    ]
    environment = str(environment_key or '').strip().casefold()
    if environment:
        exact = [
            row for row in candidates
            if row['EnvironmentKey'].casefold() == environment]
        candidates = exact or [
            row for row in candidates if not row['EnvironmentKey']]
    keys = {row['AgentKey'] for row in candidates}
    if len(keys) > 1:
        raise ValueError(
            f'Agent alias is ambiguous without an environment: {alias_type} / {alias_value}')
    return candidates[0] if candidates else None

def _raw_canonical_key(kind, value, raw_environment=''):
    normalized = normalize_agent_alias(kind, value)
    material = (
        f'{str(raw_environment or "").strip().casefold()}|'
        f'{kind.casefold()}|{normalized}')
    digest = _registry_hashlib.sha256(material.encode('utf-8')).hexdigest()[:24]
    return f'raw:{kind.casefold()}:{digest}'

def _raw_environment_key(raw_environment):
    value = str(raw_environment or '').strip()
    return _raw_canonical_key('Environment', value) if value else ''

def _clean_raw_identity(value, max_length=256):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ''
    value = str(value).strip()
    if not value or len(value) > max_length:
        return ''
    if any(char in value for char in ('{', '}', '"', '\n', '\t', '<', '>', '[', ']')):
        return ''
    return value

def _clean_raw_id(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ''
    value = str(value).strip()
    if value.startswith('{') and value.endswith('}'):
        value = value[1:-1].strip()
    return _clean_raw_identity(value)

def _extract_transcript_identity(row):
    raw_schema = _clean_raw_identity(row.get('agent_schema_hint'), 128)
    raw_id = raw_name = ''
    transcript_metadata = row.get('metadata')
    if isinstance(transcript_metadata, str) and transcript_metadata.strip():
        try:
            transcript_metadata = json.loads(transcript_metadata)
        except Exception:
            transcript_metadata = {}
    if isinstance(transcript_metadata, dict):
        raw_id = (
            _clean_raw_id(transcript_metadata.get('BotId'))
            or _clean_raw_id(transcript_metadata.get('botId')))
        raw_name = (
            _clean_raw_identity(transcript_metadata.get('BotName'))
            or _clean_raw_identity(transcript_metadata.get('botName')))
        raw_schema = (
            raw_schema
            or _clean_raw_identity(transcript_metadata.get('BotSchemaName'), 128)
            or _clean_raw_identity(transcript_metadata.get('SchemaName'), 128)
            or _clean_raw_identity(transcript_metadata.get('botSchemaName'), 128))
    nested = row.get('bot_conversationtranscriptid')
    if isinstance(nested, dict):
        raw_schema = raw_schema or _clean_raw_identity(nested.get('schemaname'), 128)
        raw_id = _clean_raw_id(nested.get('botid'))
        raw_name = _clean_raw_identity(nested.get('name'))
    for column in ('bot_conversationtranscriptid.botid', 'botid', 'bot_botid'):
        raw_id = raw_id or _clean_raw_id(row.get(column))
    for column in ('bot_conversationtranscriptid.name', 'bot_name', 'name'):
        raw_name = raw_name or _clean_raw_identity(row.get(column))
    activities = row.get('content_json')
    if isinstance(activities, dict):
        activities = next(
            (activities[key] for key in ('activities', 'Activities', 'transcript', 'messages')
             if isinstance(activities.get(key), list)), [])
    if not isinstance(activities, list):
        activities = []
    for activity in activities:
        if not isinstance(activity, dict):
            continue
        value = activity.get('value') if isinstance(activity.get('value'), dict) else {}
        raw_schema = (
            raw_schema
            or _clean_raw_identity(value.get('parentBotSchemaName'), 128)
            or _clean_raw_identity(value.get('botSchemaName'), 128))
        if activity.get('type') == 'message':
            sender = activity.get('from') if isinstance(activity.get('from'), dict) else {}
            recipient = (
                activity.get('recipient')
                if isinstance(activity.get('recipient'), dict) else {})
            bot_party = sender if sender.get('role') in (0, 'bot') else recipient
            raw_id = raw_id or _clean_raw_id(bot_party.get('id'))
            raw_name = raw_name or _clean_raw_identity(bot_party.get('name'))
    return raw_id, raw_schema, raw_name

def resolve_transcript_agent(row, registry_rows, active_keys, scope_mode):
    raw_id, raw_schema, raw_name = _extract_transcript_identity(row)
    raw_environment = _clean_raw_identity(row.get('SourceEnvironment'))
    environment_key = registry_environment_key(registry_rows, raw_environment)
    match = (
        resolve_registry_alias(
            registry_rows, 'TranscriptId', raw_id, active_keys, environment_key)
        or resolve_registry_alias(
            registry_rows, 'TranscriptSchema', raw_schema, active_keys, environment_key)
    )
    if match:
        canonical_environment = (
            match['EnvironmentKey'] or environment_key
            or _raw_environment_key(raw_environment))
        return {
            'canonical_agent_key': match['AgentKey'],
            'canonical_agent_name': match['AgentLabel'],
            'canonical_environment_key': canonical_environment,
            'agent_mapping_status': 'RegistryMapped',
            'raw_agent_id': raw_id,
            'raw_agent_schema': raw_schema,
            'raw_agent_name': raw_name,
            'raw_environment': raw_environment,
            'source_environment': raw_environment,
        }
    if scope_mode != 'All Agents':
        return None
    raw_kind, raw_value = (
        ('TranscriptId', raw_id) if raw_id
        else ('TranscriptSchema', raw_schema) if raw_schema
        else ('TranscriptName', raw_name) if raw_name
        else ('TranscriptUnknown', row.get('conversationtranscriptid')))
    return {
        'canonical_agent_key': _raw_canonical_key(
            raw_kind, raw_value, raw_environment),
        'canonical_agent_name': raw_name or raw_schema or raw_id or 'Unknown agent',
        'canonical_environment_key': (
            environment_key or _raw_environment_key(raw_environment)),
        'agent_mapping_status': 'RawAllAgents',
        'raw_agent_id': raw_id,
        'raw_agent_schema': raw_schema,
        'raw_agent_name': raw_name,
        'raw_environment': raw_environment,
        'source_environment': raw_environment,
    }

def scope_transcript_rows(frame, registry_rows, scope_mode):
    active_keys = active_agent_keys(registry_rows, scope_mode)
    scoped_rows = []
    for _, row in frame.iterrows():
        identity = resolve_transcript_agent(
            row, registry_rows, active_keys, scope_mode)
        if identity is None:
            continue
        item = row.copy()
        for key, value in identity.items():
            item[key] = value
        conversation_id = str(row.get('conversationtranscriptid') or '').strip()
        item['conversation_natural_key'] = (
            f'{identity["canonical_environment_key"]}|{conversation_id}')
        scoped_rows.append(item)
    if scope_mode != 'All Agents' and len(frame) and not scoped_rows:
        raise ValueError(
            f'No transcript agents matched AGENT_SCOPE_MODE={scope_mode!r}')
    result = pd.DataFrame(
        scoped_rows,
        columns=list(dict.fromkeys(list(frame.columns) + _IDENTITY_COLUMNS)))
    if len(result) and result['conversation_natural_key'].duplicated().any():
        duplicates = sorted(
            result.loc[
                result['conversation_natural_key'].duplicated(keep=False),
                'conversation_natural_key'].astype(str).unique().tolist())
        raise ValueError(
            'Transcript source contains duplicate environment-aware conversation keys: '
            + ', '.join(duplicates[:10]))
    return result, active_keys

AGENT_REGISTRY_ROWS = load_agent_registry(AGENT_REGISTRY_FILE)
parsed, ACTIVE_AGENT_KEYS = scope_transcript_rows(
    parsed, AGENT_REGISTRY_ROWS, AGENT_SCOPE_MODE)
print(
    f'agent registry scope: {AGENT_SCOPE_MODE}; '
    f'{len(ACTIVE_AGENT_KEYS):,} active agent(s); {len(parsed):,} scoped transcript(s)')
# === END AGENT REGISTRY RUNTIME ===


## 3. Parsing helpers

Reused verbatim from `parse_copilot_transcripts.py`. **PII handling**: raw AAD `objectId` is emitted (needed to join to `Chat + Agent Org Data`[id] for Organization slicing inside the governed Lakehouse); free-text profile fields are dropped; message bodies truncated to `TEXT_TRUNCATE` chars.

In [ ]:
def keep_user_id(v):
    """Return the raw AAD objectId unchanged so users stay joinable downstream.

    NOTE: despite the legacy output column name ``user_id_hash``, this value is NOT
    hashed - it is the raw user id, kept intentionally for cross-source joins.
    Empty -> None so joins stay clean.
    """
    if v is None or v == '':
        return None
    return str(v)

# Backwards-compatible alias (existing cells call hash_id); same pass-through, no hashing.
hash_id = keep_user_id

def get_activities(content):
    if isinstance(content, dict):
        for k in ('activities', 'Activities', 'transcript', 'messages'):
            if k in content and isinstance(content[k], list):
                return content[k]
    if isinstance(content, list):
        return content
    return []

def extract_agent_name(act_list):
    """Returns (primary_agent_schema, [connected_agent_schemas...])."""
    primary = None
    connected = []
    for a in act_list:
        if not isinstance(a, dict):
            continue
        v = a.get('value') or {}
        if not isinstance(v, dict):
            continue
        if a.get('valueType') == 'ConnectedAgentInitializeTraceData':
            sn = v.get('connectedAgentBotSchemaName') or v.get('botSchemaName')
            if sn and sn not in connected:
                connected.append(sn)
            if primary is None:
                primary = v.get('parentBotSchemaName')
        # Single-agent transcripts have no ConnectedAgent trace; pick up a botSchemaName anywhere.
        elif primary is None:
            bsn = v.get('botSchemaName') or v.get('parentBotSchemaName')
            if bsn:
                primary = bsn
    return primary, connected

def _verdict_from_value(val):
    if val is None:
        return None
    if isinstance(val, bool):
        return 'Positive' if val else 'Negative'
    if isinstance(val, (int, float)):
        return 'Positive' if val > 0 else 'Negative'
    s = str(val).strip().lower()
    if s in ('true','1','like','thumbsup','thumbs up','up','positive','helpful','yes','good','smile'):
        return 'Positive'
    if s in ('false','0','dislike','thumbsdown','thumbs down','down','negative','unhelpful','no','bad','frown'):
        return 'Negative'
    return None

def extract_submitted_feedback(act_list):
    """Scan every canonical schema position where a completed feedback verdict
    writes back into a transcript activity stream. Returns (verdict, comment)."""
    verdict = None
    comment = None
    for a in act_list:
        if not isinstance(a, dict):
            continue
        for rx in (a.get('reactionsAdded') or []):
            if isinstance(rx, dict):
                v = _verdict_from_value(rx.get('type') or rx.get('reactionType'))
                if v:
                    verdict = v
        cd = a.get('channelData') if isinstance(a.get('channelData'), dict) else {}
        fl = cd.get('feedbackLoop') if isinstance(cd.get('feedbackLoop'), dict) else {}
        if fl:
            v = _verdict_from_value(fl.get('value') or fl.get('reaction') or fl.get('vote'))
            if v:
                verdict = v
            comment = comment or fl.get('feedbackText') or fl.get('text')
        cf = cd.get('feedback') if isinstance(cd.get('feedback'), dict) else {}
        if cf:
            v = _verdict_from_value(cf.get('helpful') if 'helpful' in cf else cf.get('reaction'))
            if v:
                verdict = v
            comment = comment or cf.get('feedbackText') or cf.get('text')
        # Canonical messageReaction activity writes reaction/comment under value.
        if a.get('type') == 'messageReaction':
            reaction_value = a.get('value') if isinstance(a.get('value'), dict) else {}
            v = _verdict_from_value(
                reaction_value.get('reaction')
                or reaction_value.get('reactionType')
                or a.get('reaction'))
            if v:
                verdict = v
            comment = (
                comment
                or reaction_value.get('comment')
                or reaction_value.get('feedbackText')
                or reaction_value.get('text'))
        val = a.get('value') if isinstance(a.get('value'), dict) else {}
        if val:
            action = str(val.get('actionName') or a.get('name') or '').lower()
            fb = val.get('feedback') if isinstance(val.get('feedback'), dict) else None
            if fb or 'feedback' in action or 'submitaction' in action:
                src = fb or val
                v = _verdict_from_value(src.get('helpful') if 'helpful' in src else (src.get('reaction') or src.get('vote') or src.get('rating')))
                if v:
                    verdict = v
                comment = comment or src.get('feedbackText') or src.get('text')
    return verdict, comment

def _iso(ms):
    return datetime.fromtimestamp(ms/1000, tz=timezone.utc).isoformat() if ms else None

def _clean_token(val, maxlen=200):
    """Clean a candidate agent label/id. Unlike _clean_schema this ALLOWS spaces
    (display names are multi-word) but still rejects CSV-corruption bleed."""
    if val is None:
        return None
    s = str(val).strip()
    if not s or len(s) > maxlen:
        return None
    if any(ch in s for ch in ('{', '}', '"', '\n', '\t', '<', '>', '[', ']')):
        return None
    return s

def resolve_agent_identity(acts, row):
    """Resolve (agent_schema, agent_name, agent_id, connected_schemas) for a transcript
    using the most reliable signal available, in priority order:
      1. Dataverse bot lookup  (schemaname / name / botid)        - survives corrupt content
      2. Bot-message from.name / from.id                          - present in almost every convo
      3. User-message recipient.name / recipient.id               - the agent as addressed
      4. Content orchestration trace botSchemaName / parentBot... - multi-agent traces only
      5. Transcript-level `name` field                            - last-resort label

    Why this exists: the previous logic resolved identity ONLY from the bot-lookup
    schemaname plus connected-agent orchestration traces. Single-agent transcripts have
    neither, so primary_agent_schema was null for the bulk of conversations -> sessions
    unattributed and the self-sourced agent_catalogue collapsed to a handful of agents.
    The bot's from/recipient identity is present in nearly every conversation, so we use it.
    """
    # --- connected (sub-)agents + any parent/bot schema from orchestration traces ---
    connected, trace_schema = [], None
    for a in acts:
        if not isinstance(a, dict):
            continue
        v = a.get('value') or {}
        if not isinstance(v, dict):
            continue
        if a.get('valueType') == 'ConnectedAgentInitializeTraceData':
            sn = v.get('connectedAgentBotSchemaName') or v.get('botSchemaName')
            if sn and sn not in connected:
                connected.append(sn)
            if trace_schema is None:
                trace_schema = v.get('parentBotSchemaName')
        elif trace_schema is None:
            bsn = v.get('botSchemaName') or v.get('parentBotSchemaName')
            if bsn:
                trace_schema = bsn

    # --- bot lookup (nested dict from $expand, or flat columns from a CSV export) ---
    lookup_schema = row.get('agent_schema_hint')  # pre-cleaned schema token (cell 2b)
    lookup_name = lookup_id = None
    nbref = row.get('bot_conversationtranscriptid')
    if isinstance(nbref, dict):
        lookup_schema = lookup_schema or _clean_schema(nbref.get('schemaname'))
        lookup_name = _clean_token(nbref.get('name'))
        lookup_id = _clean_token(nbref.get('botid'))
    for c in ('bot_conversationtranscriptid.name', 'bot_name'):
        if not lookup_name and c in row:
            lookup_name = _clean_token(row.get(c))
    for c in ('bot_conversationtranscriptid.botid', 'botid', 'bot_botid'):
        if not lookup_id and c in row:
            lookup_id = _clean_token(row.get(c))

    # --- from/recipient identity on the message stream ---
    msg_name = msg_id = None
    for a in acts:
        if not isinstance(a, dict) or a.get('type') != 'message':
            continue
        frm = a.get('from') or {}
        if frm.get('role') in (0, 'bot'):
            msg_name = msg_name or _clean_token(frm.get('name'))
            msg_id = msg_id or _clean_token(frm.get('id'))
        else:
            rcp = a.get('recipient') or {}
            if isinstance(rcp, dict):
                msg_name = msg_name or _clean_token(rcp.get('name'))
                msg_id = msg_id or _clean_token(rcp.get('id'))
        if msg_name and msg_id:
            break

    tx_name = _clean_token(row.get('name'))

    schema = lookup_schema or trace_schema
    name = lookup_name or msg_name or tx_name
    # __ID_GUARD_RESOLVE_NAME__ : never surface a raw Teams/thread identity id as the agent name
    if name and re.search(r'(@thread\.|@unq\.gbl|^(?:8|19|28|29|48):|^a:1|[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12})', str(name), re.I):
        name = None
    bot_id = lookup_id or msg_id
    # Keep the relationship key (agent_schema) populated even when no schema token exists,
    # so the session is attributed and the agent appears in the catalogue. Prefer a human
    # name over a raw id for the fallback key (the catalogue humanises it for display).
    if not schema:
        # __ID_GUARD_FALLBACK__ : don't let a raw identity id become the agent key
        cand = name or bot_id
        schema = cand if (cand and not re.search(r'(@thread\.|@unq\.gbl|^(?:8|19|28|29|48):|^a:1|[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12})', str(cand), re.I)) else None

    return schema, name, bot_id, connected

print('helpers loaded')

# === CANONICAL IDENTITY HELPERS (generated) ===
def _identity_fields(row):
    return {column: row.get(column) for column in _IDENTITY_COLUMNS}
# === END CANONICAL IDENTITY HELPERS ===


## 4. Build per-session fact table — `agent_sessions`

In [ ]:
def build_sessions(parsed):
    rows = []
    for _, r in parsed.iterrows():
        acts = get_activities(r['content_json'])
        msgs = [a for a in acts if isinstance(a, dict) and a.get('type') == 'message']
        user_msgs = [a for a in msgs if (a.get('from') or {}).get('role') in (1, 'user')]
        bot_msgs  = [a for a in msgs if (a.get('from') or {}).get('role') in (0, 'bot')]
        user_id_raw = None
        for a in msgs:
            if (a.get('from') or {}).get('role') in (1, 'user'):
                user_id_raw = (a.get('from') or {}).get('aadObjectId') or (a.get('from') or {}).get('id')
                break
        primary_agent, agent_name, agent_bot_id, connected_agents = resolve_agent_identity(acts, r)
        cost = latency = plan_steps = 0
        for a in acts:
            v = a.get('value') if isinstance(a, dict) else None
            if isinstance(v, dict) and 'displayedCost' in v:
                try:
                    cost += int(v.get('displayedCost') or 0)
                except (TypeError, ValueError):
                    pass
                plan_steps += 1
                et = v.get('executionTime')
                if et:
                    m = re.match(r'(\d+):(\d+):(\d+\.?\d*)', str(et))
                    if m:
                        h, mn, s = int(m.group(1)), int(m.group(2)), float(m.group(3))
                        latency += int((h*3600 + mn*60 + s) * 1000)
        k_searched = k_answered = False
        k_sources = 0
        for a in acts:
            if isinstance(a, dict) and a.get('valueType') == 'KnowledgeTraceData':
                v = a.get('value') or {}
                if isinstance(v, dict):
                    if v.get('isKnowledgeSearched'):
                        k_searched = True
                    if v.get('completionState') == 'Answered':
                        k_answered = True
                    k_sources += len(v.get('citedKnowledgeSources') or [])
        # --- grounding source: did the agent answer from internal knowledge, the web, or neither ---
        g_internal = g_external = False
        for a in acts:
            if not isinstance(a, dict):
                continue
            vt = a.get('valueType'); nm = str(a.get('name') or ''); gv = a.get('value') or {}
            if vt == 'KnowledgeTraceData' and isinstance(gv, dict) and gv.get('isKnowledgeSearched'):
                g_internal = True
            if 'UniversalSearch' in nm and isinstance(gv, dict):
                if gv.get('knowledgeSources'):
                    g_internal = True
                for ep in ((gv.get('searchContextTraceInfo') or {}).get('endpoints') or []):
                    h = str(ep).lower()
                    if 'bing.com' in h or 'bing.net' in h:
                        g_external = True
                    elif 'sharepoint' in h or 'onedrive' in h or 'crm.dynamics' in h or 'graph.microsoft' in h:
                        g_internal = True
                    elif h.startswith('http'):
                        g_external = True
            if 'web' in nm.lower() and 'search' in nm.lower():
                g_external = True
        grounding_source = ('Mixed (Internal+Web)' if g_internal and g_external
                            else 'External (Web)' if g_external
                            else 'Internal' if g_internal
                            else 'Ungrounded')
        errors = [ (a.get('value') or {}).get('errorCode') or 'Unknown'
                   for a in acts if isinstance(a, dict) and a.get('valueType') == 'ErrorTraceData' ]
        feedback_offered = sum(1 for a in acts if isinstance(a, dict)
                               and isinstance(a.get('channelData'), dict)
                               and 'feedbackLoop' in a.get('channelData', {}))
        verdict, comment = extract_submitted_feedback(acts)
        # Authoritative Copilot Studio SessionInfo outcome.
        session_info_values = [
            a.get('value') or {}
            for a in acts
            if isinstance(a, dict)
            and (a.get('valueType') == 'SessionInfo' or a.get('name') == 'SessionInfo')
            and isinstance(a.get('value'), dict)]
        session_info = session_info_values[-1] if session_info_values else {}
        session_outcome_raw = str(
            session_info.get('outcome')
            or session_info.get('lastSessionOutcome')
            or '').strip()
        session_outcome_map = {
            'resolved': 'Resolved',
            'escalated': 'Escalated',
            'abandon': 'Abandoned',
            'abandoned': 'Abandoned',
        }
        session_outcome_explicit = session_outcome_map.get(
            session_outcome_raw.lower(), session_outcome_raw or None)
        session_outcome_reason_explicit = (
            session_info.get('outcomeReason')
            or session_info.get('lastSessionOutcomeReason'))
        first_ts = last_ts = None
        for a in acts:
            ts = a.get('timestampMs') if isinstance(a, dict) else None
            if ts:
                first_ts = ts if first_ts is None or ts < first_ts else first_ts
                last_ts  = ts if last_ts  is None or ts > last_ts  else last_ts
        duration_ms = (last_ts - first_ts) if first_ts and last_ts else None
        session_start = _iso(first_ts) or r.get('conversationstarttime', '')
        first_prompt = ''
        for a in user_msgs:
            if a.get('text', ''):
                first_prompt = a['text'][:TEXT_TRUNCATE]
                break
        session_locale = ''
        for a in acts:
            if not isinstance(a, dict):
                continue
            if a.get('name') == 'pvaSetContext':
                session_locale = str((a.get('value') or {}).get('locale') or '')
                if session_locale:
                    break
            if not session_locale and a.get('locale'):
                session_locale = str(a.get('locale'))
        if not session_locale:
            session_locale = str(r.get('locale') or '')
        _native_noise = {
            'logging', 'greeting', 'goodbye', 'end of conversation', 'start over',
            'escalate', 'sign in', 'signin', 'reset conversation', 'on error',
            'multiple topics matched', 'fallback', 'conversational boosting',
            'thumbs up', 'thumbs down'}
        _native_topics = []
        for a in acts:
            if not isinstance(a, dict):
                continue
            if a.get('valueType') != 'IntentRecognition' and a.get('name') != 'IntentRecognition':
                continue
            title = str((a.get('value') or {}).get('intentTitle') or '').strip()
            if (title and not title.startswith(('[System]', '[Internal]', '[Admin]', '[Debug]', '[Test]'))
                    and title.lower() not in _native_noise):
                _native_topics.append(title)
        primary_topic_native = _native_topics[0] if _native_topics else None
        rows.append({
            **_identity_fields(r),
            'conversation_id': r.get('conversationtranscriptid', ''),
            'session_start_utc': session_start,
            'session_duration_ms': duration_ms,
            'user_id_hash': hash_id(user_id_raw),
            'primary_agent_schema': primary_agent,
            'connected_agent_schemas': '|'.join(connected_agents) if connected_agents else None,
            'connected_agent_count': len(connected_agents),
            'agent_name': agent_name,
            'agent_id': agent_bot_id,
            'msg_count': len(msgs),
            'user_msg_count': len(user_msgs),
            'bot_msg_count': len(bot_msgs),
            'plan_step_count': plan_steps,
            'total_displayed_cost': cost,
            'total_latency_ms': latency,
            'knowledge_searched': k_searched,
            'knowledge_answered': k_answered,
            'knowledge_sources_count': k_sources,
            'grounding_source': grounding_source,
            'error_count': len(errors),
            'first_error_code': errors[0] if errors else None,
            'feedback_offered_count': feedback_offered,
            'feedback_submitted': 1 if verdict else 0,
            'feedback_verdict': verdict,
            'feedback_comment': comment,
            'locale': session_locale,
            'primary_topic_native': primary_topic_native,
            'first_user_prompt': first_prompt,
            # Preserve the authoritative SessionInfo outcome; downstream heuristics apply only when absent.
            'session_outcome_explicit': session_outcome_explicit,
            'session_outcome_reason_explicit': session_outcome_reason_explicit,
            'is_authenticated': bool(user_id_raw),
            'is_returning_user': None,        # filled in below (user appears in >1 conversation)
            'primary_topic_derived': None,
        })
    _df = pd.DataFrame(rows, columns=_IDENTITY_COLUMNS + ['conversation_id', 'session_start_utc', 'session_duration_ms', 'user_id_hash', 'primary_agent_schema', 'connected_agent_schemas', 'connected_agent_count', 'agent_name', 'agent_id', 'msg_count', 'user_msg_count', 'bot_msg_count', 'plan_step_count', 'total_displayed_cost', 'total_latency_ms', 'knowledge_searched', 'knowledge_answered', 'knowledge_sources_count', 'grounding_source', 'error_count', 'first_error_code', 'feedback_offered_count', 'feedback_submitted', 'feedback_verdict', 'feedback_comment', 'locale', 'primary_topic_native', 'first_user_prompt', 'session_outcome_explicit', 'session_outcome_reason_explicit', 'is_authenticated', 'is_returning_user', 'primary_topic_derived'])
    # is_returning_user: a user_id_hash that appears in more than one conversation.
    _vc = _df['user_id_hash'].value_counts()
    _returning = set(_vc[_vc > 1].index) - {None, ''}
    _df['is_returning_user'] = _df['user_id_hash'].apply(
        lambda u: True if (u is not None and u != '' and u in _returning) else False)
    return _df

sessions = build_sessions(parsed)
print(f'sessions: {len(sessions):,}')

## 5. Build per-turn fact table — `agent_turns`

In [ ]:
def build_turns(parsed):
    rows = []
    for _, r in parsed.iterrows():
        conv_id = r.get('conversationtranscriptid', '')
        acts = get_activities(r['content_json'])
        intent_by_reply, knowledge_by_reply = {}, {}
        for a in acts:
            if isinstance(a, dict) and a.get('valueType') == 'IntentRecognition':
                v = a.get('value') or {}
                rid = a.get('replyToId')
                if rid and isinstance(v, dict):
                    intent_by_reply[rid] = {
                        'intent_title': v.get('intentTitle'),
                        'intent_id': v.get('intentId'),
                        'intent_score': (v.get('intentScore') or {}).get('score'),
                    }
            if isinstance(a, dict) and a.get('valueType') == 'KnowledgeTraceData':
                v = a.get('value') or {}
                rid = a.get('replyToId')
                if rid and isinstance(v, dict):
                    knowledge_by_reply[rid] = {
                        'knowledge_searched': bool(v.get('isKnowledgeSearched')),
                        'knowledge_answered': v.get('completionState') == 'Answered',
                        'knowledge_sources': '|'.join(v.get('citedKnowledgeSources') or []),
                    }
        message_ordinal = 0
        for a in acts:
            if not isinstance(a, dict) or a.get('type') != 'message':
                continue
            role = (a.get('from') or {}).get('role')
            role_label = 'user' if role in (1, 'user') else ('bot' if role in (0, 'bot') else 'other')
            ts_ms = a.get('timestampMs')
            from_id = (a.get('from') or {}).get('aadObjectId') or (a.get('from') or {}).get('id')
            msg_id = a.get('id') or f'{conv_id}:message:{message_ordinal}'
            message_ordinal += 1
            intent = intent_by_reply.get(msg_id, {})
            knowledge = knowledge_by_reply.get(msg_id, {})
            cd = a.get('channelData') or {}
            fb_offered = isinstance(cd, dict) and 'feedbackLoop' in cd
            turn_verdict, turn_comment = extract_submitted_feedback([a])
            rows.append({
                **_identity_fields(r),
                'conversation_id': conv_id,
                'turn_id': msg_id,
                'turn_timestamp_utc': _iso(ts_ms),
                'turn_role': role_label,
                'turn_channel': a.get('channelId'),
                'from_user_hash': hash_id(from_id) if role_label == 'user' else None,
                'text_preview_500': (a.get('text') or '')[:TEXT_TRUNCATE],
                'text_length': len(a.get('text') or ''),
                'intent_title': intent.get('intent_title'),
                'intent_id': intent.get('intent_id'),
                'intent_score': intent.get('intent_score'),
                'knowledge_searched': knowledge.get('knowledge_searched'),
                'knowledge_answered': knowledge.get('knowledge_answered'),
                'knowledge_sources': knowledge.get('knowledge_sources'),
                'feedback_offered': fb_offered,
                'feedback_verdict': turn_verdict,
                'feedback_comment': turn_comment,
            })
    return pd.DataFrame(rows, columns=_IDENTITY_COLUMNS + ['conversation_id', 'turn_id', 'turn_timestamp_utc', 'turn_role', 'turn_channel', 'from_user_hash', 'text_preview_500', 'text_length', 'intent_title', 'intent_id', 'intent_score', 'knowledge_searched', 'knowledge_answered', 'knowledge_sources', 'feedback_offered', 'feedback_verdict', 'feedback_comment'])

turns = build_turns(parsed)
print(f'turns: {len(turns):,}')

## 6. Errors + sub-agent invocations — `agent_errors`, `agent_subagents`

In [ ]:
def build_errors(parsed):
    rows = []
    for _, r in parsed.iterrows():
        for a in get_activities(r['content_json']):
            if not isinstance(a, dict) or a.get('valueType') != 'ErrorTraceData':
                continue
            v = a.get('value') or {}
            rows.append({
                **_identity_fields(r),
                'conversation_id': r.get('conversationtranscriptid', ''),
                'error_timestamp_utc': _iso(a.get('timestampMs')),
                'error_code': v.get('errorCode'),
                'error_sub_code': v.get('errorSubCode'),
                'error_message': (v.get('errorMessage') or '')[:TEXT_TRUNCATE],
                'is_user_error': bool(v.get('isUserError')),
            })
    return pd.DataFrame(rows, columns=_IDENTITY_COLUMNS + ['conversation_id', 'error_timestamp_utc', 'error_code', 'error_sub_code', 'error_message', 'is_user_error'])

def build_subagents(parsed):
    rows = []
    for _, r in parsed.iterrows():
        child_event_sequence = 0
        for a in get_activities(r['content_json']):
            if not isinstance(a, dict):
                continue
            vt = a.get('valueType')
            if vt not in ('ConnectedAgentInitializeTraceData', 'ConnectedAgentCompletedTraceData'):
                continue
            child_event_sequence += 1
            v = a.get('value') or {}
            event_type = 'initialize' if vt.endswith('InitializeTraceData') else 'completed'
            connected_schema = v.get('connectedAgentBotSchemaName') or v.get('botSchemaName')
            connected_id = v.get('connectedAgentBotId') or v.get('connectedAgentId') or v.get('botId')
            connected_name = v.get('connectedAgentName') or v.get('agentName') or v.get('botName')
            plan_step_id = v.get('planStepId')
            activity_id = a.get('id') or a.get('activityId')
            timestamp_raw = a.get('timestampMs') or a.get('timestamp')
            agent_identity = connected_schema or connected_id or connected_name or 'unknown'
            subagent_event_key = (
                f'activity:{activity_id}' if activity_id
                else f'sequence:{child_event_sequence}|{event_type}|'
                     f'{timestamp_raw or plan_step_id or agent_identity}')
            completion_state = (
                v.get('completionState') or v.get('status') or v.get('outcome')
                or ('Initialized' if event_type == 'initialize' else 'Completed'))
            rows.append({
                **_identity_fields(r),
                'conversation_id': r.get('conversationtranscriptid', ''),
                'subagent_event_key': subagent_event_key,
                'invocation_timestamp_utc': _iso(a.get('timestampMs') or a.get('timestamp')),
                'event_type': event_type,
                'parent_agent_schema': v.get('parentBotSchemaName'),
                'parent_agent_id': v.get('parentBotId') or v.get('parentAgentId'),
                'parent_agent_name': v.get('parentAgentName') or v.get('parentBotName'),
                'connected_agent_schema': connected_schema,
                'connected_agent_id': connected_id,
                'connected_agent_name': connected_name,
                'dialog_schema': v.get('dialogSchemaName') or v.get('dialogName'),
                'user_id_hash': hash_id(v.get('userId')),
                'plan_step_id': plan_step_id,
                'invocation_state': completion_state,
                'completion_state': completion_state if event_type == 'completed' else None,
                'error_code': v.get('errorCode'),
                'error_message': (v.get('errorMessage') or '')[:TEXT_TRUNCATE],
            })
    return pd.DataFrame(rows, columns=_IDENTITY_COLUMNS + [
        'conversation_id', 'subagent_event_key', 'invocation_timestamp_utc', 'event_type',
        'parent_agent_schema', 'parent_agent_id', 'parent_agent_name',
        'connected_agent_schema', 'connected_agent_id', 'connected_agent_name',
        'dialog_schema', 'user_id_hash', 'plan_step_id', 'invocation_state',
        'completion_state', 'error_code', 'error_message'])

def summarize_subagents(subagent_frame):
    columns = [
        'conversation_natural_key', 'parent_agent_name', 'parent_agent_id',
        'child_agent_names', 'child_agent_schema_names', 'child_agent_ids',
        'child_agent_invocation_states', 'child_agent_dialogs', 'child_agent_plan_steps',
        'child_agent_invocation_count', 'child_agent_count', 'has_child_agent',
        'child_agent_detail']
    if subagent_frame.empty:
        return pd.DataFrame(columns=columns)

    def _present(value):
        return pd.notna(value) and str(value).strip() != ''

    def _first_present(*values):
        return next((value for value in values if _present(value)), None)

    def _join(values):
        return ' | '.join(dict.fromkeys(str(value) for value in values if _present(value)))

    summaries = []
    for conversation_key, group in subagent_frame.groupby(
            'conversation_natural_key', dropna=False):
        group = group.copy()
        first = group.iloc[0]
        initialize_sequence = completed_sequence = 0
        fallback_keys = []
        for event_type in group['event_type'].tolist():
            if event_type == 'initialize':
                initialize_sequence += 1
                fallback_keys.append(f'sequence:{initialize_sequence}')
            else:
                completed_sequence += 1
                fallback_keys.append(f'sequence:{completed_sequence}')
        group['_fallback_invocation_key'] = fallback_keys

        def _invocation_key(row):
            plan_step = _first_present(row.get('plan_step_id'))
            return f'plan:{plan_step}' if plan_step else row.get('_fallback_invocation_key')

        invocation_keys = list(dict.fromkeys(
            key for key in group.apply(_invocation_key, axis=1).tolist() if key))
        identities, details = [], []
        for invocation_key in invocation_keys:
            matching = group[group.apply(_invocation_key, axis=1) == invocation_key]
            identity = _first_present(
                *matching['connected_agent_schema'].tolist(),
                *matching['connected_agent_id'].tolist(),
                *matching['connected_agent_name'].tolist())
            if identity and identity not in identities:
                identities.append(identity)
            completed = matching[matching['event_type'] == 'completed']
            chosen = completed.iloc[-1] if not completed.empty else matching.iloc[-1]
            names = [
                value for value in matching['connected_agent_name'].tolist()
                if _present(value)]
            label = names[-1] if names else _first_present(
                chosen.get('connected_agent_schema'),
                chosen.get('connected_agent_id'), identity)
            details.append(
                f'{label} ({chosen.get("invocation_state") or chosen.get("event_type")})')
        summaries.append({
            'conversation_natural_key': conversation_key,
            'parent_agent_name': first.get('parent_agent_name'),
            'parent_agent_id': first.get('parent_agent_id'),
            'child_agent_names': _join(group['connected_agent_name']),
            'child_agent_schema_names': _join(group['connected_agent_schema']),
            'child_agent_ids': _join(group['connected_agent_id']),
            'child_agent_invocation_states': _join(group['invocation_state']),
            'child_agent_dialogs': _join(group['dialog_schema']),
            'child_agent_plan_steps': _join(group['plan_step_id']),
            'child_agent_invocation_count': len(invocation_keys),
            'child_agent_count': len(identities),
            'has_child_agent': bool(identities),
            'child_agent_detail': '; '.join(details),
        })
    return pd.DataFrame(summaries, columns=columns)

errors = build_errors(parsed)
subagents = build_subagents(parsed)
child_summary = summarize_subagents(subagents)
_session_child_columns = {
    'parent_agent_name': None,
    'parent_agent_id': None,
    'child_agent_names': None,
    'child_agent_schema_names': None,
    'child_agent_ids': None,
    'child_agent_invocation_states': None,
    'child_agent_dialogs': None,
    'child_agent_plan_steps': None,
    'child_agent_invocation_count': 0,
    'child_agent_count': 0,
    'has_child_agent': False,
    'child_agent_detail': None,
}
if not child_summary.empty:
    sessions = sessions.merge(
        child_summary, on='conversation_natural_key', how='left',
        validate='one_to_one')
for _column, _default in _session_child_columns.items():
    if _column not in sessions.columns:
        sessions[_column] = _default
    else:
        sessions[_column] = sessions[_column].where(pd.notna(sessions[_column]), _default)
print(f'errors: {len(errors):,}  |  sub-agent events: {len(subagents):,}')


## 7. Agent catalogue (dimension) — `agent_catalogue`

In [ ]:
# Registry-backed conformed agent dimension.
def build_agent_dim(session_frame, registry_rows):
    columns = [
        'canonical_agent_key', 'canonical_agent_name', 'canonical_environment_key',
        'agent_schema', 'agent_display_name', 'agent_class', 'raw_agent_id',
        'raw_agent_schema', 'raw_agent_name', 'agent_mapping_status',
        'source_environments']
    if session_frame.empty:
        return pd.DataFrame(columns=columns)
    metadata = {}
    for row in registry_rows:
        metadata.setdefault(row['AgentKey'], row)
    records = []
    for agent_key, group in session_frame.groupby(
            'canonical_agent_key', dropna=False, sort=True):
        first = group.iloc[0]
        registry = metadata.get(agent_key)
        environments = sorted({
            str(value) for value in group['canonical_environment_key']
            if pd.notna(value) and str(value)})
        source_environments = sorted({
            str(value) for value in group['source_environment']
            if pd.notna(value) and str(value)})
        agent_name = (
            registry['AgentLabel'] if registry
            else str(first.get('canonical_agent_name') or agent_key))
        records.append({
            'canonical_agent_key': agent_key,
            'canonical_agent_name': agent_name,
            'canonical_environment_key': '|'.join(environments),
            'agent_schema': first.get('raw_agent_schema'),
            'agent_display_name': agent_name,
            'agent_class': (
                'ESS' if registry and registry['IsEss']
                else 'Registry' if registry else 'Raw (All Agents)'),
            'raw_agent_id': first.get('raw_agent_id'),
            'raw_agent_schema': first.get('raw_agent_schema'),
            'raw_agent_name': first.get('raw_agent_name'),
            'agent_mapping_status': first.get('agent_mapping_status'),
            'source_environments': '|'.join(source_environments),
        })
    result = pd.DataFrame(records, columns=columns)
    if result['canonical_agent_key'].isna().any():
        raise ValueError('Agent catalogue contains a null canonical_agent_key')
    if result['canonical_agent_key'].duplicated().any():
        raise ValueError('Agent catalogue contains duplicate canonical_agent_key values')
    return result

agent_dim = build_agent_dim(sessions, AGENT_REGISTRY_ROWS)
print(f'unique canonical agents: {len(agent_dim):,}')


## 7b. Registry-backed agent scope

Scoping is applied before any transcript-derived table is built.


In [ ]:
# Agent scope is applied to parsed transcripts before any facts or dimensions are built.
if len(sessions) and not sessions['canonical_agent_key'].notna().all():
    raise ValueError('Scoped sessions contain a missing canonical_agent_key')
print(f'agent registry scope already applied: sessions={len(sessions):,}, subagents={len(subagents):,}')


## 7c. Shared multilingual topic classification

Runs fully inside Fabric. Stage 1 uses the same customer-neutral taxonomy and scoring contract as CSV/Dataverse. Stage 2 groups remaining uncategorized prompts with multilingual character n-grams and labels them as auto-discovered candidates.


In [ ]:
# === 7c-1. SHARED MULTILINGUAL TOPIC CLASSIFICATION (offline) ===
# Generated from taxonomy/topics-taxonomy.csv. No transcript data leaves Fabric.
TOPIC_CLASSIFIER_VERSION = 'ESS-TOPICS-2026.10'
TOPIC_TAXONOMY = [
    {
        "locale": "*",
        "vertical": "ESS Agent",
        "topic": "Agent Help & Capabilities",
        "keywords": [
            "employee self-service agent",
            "employee self service",
            "self-service agent",
            "what can you do",
            "how do you work",
            "example prompts",
            "agent capabilities"
        ],
        "exclusions": [],
        "priority": 95,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "IT",
        "topic": "IT Ticketing & Incidents",
        "keywords": [
            "it ticket",
            "support ticket",
            "service ticket",
            "ticket",
            "tickets",
            "incident",
            "incidents",
            "service desk",
            "help desk",
            "helpdesk",
            "servicenow",
            "service now",
            "open tickets",
            "my tickets"
        ],
        "exclusions": [],
        "priority": 95,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "IT",
        "topic": "Accounts & Password",
        "keywords": [
            "password",
            "passkey",
            "password reset",
            "reset my password",
            "unlock account",
            "locked out",
            "mfa",
            "multi-factor authentication",
            "authenticator",
            "sign in",
            "signin",
            "login",
            "credentials"
        ],
        "exclusions": [],
        "priority": 90,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "IT",
        "topic": "Access & Permissions",
        "keywords": [
            "request access",
            "grant access",
            "permissions",
            "permission",
            "admin rights",
            "security group",
            "distribution group",
            "shared drive access",
            "application access"
        ],
        "exclusions": [],
        "priority": 86,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "IT",
        "topic": "Enterprise Apps & Systems",
        "keywords": [
            "workday",
            "concur",
            "sap",
            "oracle",
            "ariba",
            "virtual desktop",
            "power platform",
            "copilot studio",
            "enterprise application"
        ],
        "exclusions": [],
        "priority": 82,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "IT",
        "topic": "Software & Downloads",
        "keywords": [
            "software",
            "application install",
            "install application",
            "install software",
            "software license",
            "application license",
            "download software",
            "microsoft 365",
            "office 365",
            "adobe",
            "docusign"
        ],
        "exclusions": [],
        "priority": 78,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "IT",
        "topic": "Device & Hardware",
        "keywords": [
            "laptop",
            "desktop computer",
            "computer replacement",
            "monitor",
            "printer",
            "keyboard",
            "mouse",
            "headset",
            "webcam",
            "hardware",
            "company phone",
            "mobile device",
            "intune",
            "company portal"
        ],
        "exclusions": [],
        "priority": 80,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "IT",
        "topic": "Network & Connectivity",
        "keywords": [
            "vpn",
            "wifi",
            "wi-fi",
            "network connection",
            "internet connection",
            "proxy",
            "remote access",
            "network outage",
            "connectivity"
        ],
        "exclusions": [],
        "priority": 80,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "IT",
        "topic": "Productivity How-To",
        "keywords": [
            "excel help",
            "in excel",
            "freeze panes",
            "taskbar",
            "write an email",
            "format a document",
            "microsoft office help"
        ],
        "exclusions": [],
        "priority": 60,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "HR",
        "topic": "Benefits & Retirement",
        "keywords": [
            "benefits",
            "health insurance",
            "medical plan",
            "dental plan",
            "vision plan",
            "insurance coverage",
            "401k",
            "pension",
            "hsa",
            "fsa",
            "open enrollment",
            "beneficiary",
            "cobra"
        ],
        "exclusions": [],
        "priority": 90,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "HR",
        "topic": "Disability & Leave of Absence",
        "keywords": [
            "short term disability",
            "long term disability",
            "leave of absence",
            "fmla",
            "workers compensation",
            "workers comp",
            "jury duty",
            "bereavement leave"
        ],
        "exclusions": [],
        "priority": 88,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "HR",
        "topic": "Payroll & Compensation",
        "keywords": [
            "payroll",
            "paycheck",
            "pay date",
            "payday",
            "salary",
            "direct deposit",
            "w-2",
            "w2",
            "paystub",
            "pay stub",
            "compensation",
            "bonus",
            "tax withholding"
        ],
        "exclusions": [],
        "priority": 88,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "HR",
        "topic": "Time Off & Attendance",
        "keywords": [
            "pto",
            "vacation",
            "holiday",
            "sick day",
            "sick leave",
            "time off",
            "annual leave",
            "attendance",
            "timesheet",
            "time sheet",
            "time entry",
            "parental leave",
            "maternity leave",
            "paternity leave"
        ],
        "exclusions": [],
        "priority": 86,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "HR",
        "topic": "Policy & Handbook",
        "keywords": [
            "hr policy",
            "employee policy",
            "handbook",
            "code of conduct",
            "remote work policy",
            "dress code",
            "employment verification",
            "company policy"
        ],
        "exclusions": [],
        "priority": 74,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "HR",
        "topic": "Onboarding & Offboarding",
        "keywords": [
            "onboarding",
            "new hire",
            "orientation",
            "first day",
            "offboarding",
            "resignation",
            "last day",
            "exit process",
            "termination process"
        ],
        "exclusions": [],
        "priority": 80,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "HR",
        "topic": "Learning & Development",
        "keywords": [
            "training",
            "course",
            "learning",
            "mentorship",
            "tuition reimbursement",
            "education reimbursement",
            "professional development",
            "certification program"
        ],
        "exclusions": [],
        "priority": 72,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "HR",
        "topic": "Recruiting & Referrals",
        "keywords": [
            "employee referral",
            "job posting",
            "open position",
            "recruiting",
            "resume",
            "candidate",
            "hiring process"
        ],
        "exclusions": [],
        "priority": 68,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "Facilities",
        "topic": "Building & Access",
        "keywords": [
            "building access",
            "badge access",
            "door access",
            "access card",
            "security pass",
            "visitor access",
            "office entry",
            "turnstile"
        ],
        "exclusions": [],
        "priority": 84,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "Facilities",
        "topic": "Room & Desk Booking",
        "keywords": [
            "book a room",
            "reserve a room",
            "meeting room",
            "conference room",
            "desk booking",
            "hot desk",
            "workspace reservation",
            "parking reservation"
        ],
        "exclusions": [],
        "priority": 80,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "Facilities",
        "topic": "Maintenance & Repair",
        "keywords": [
            "facilities maintenance",
            "maintenance request",
            "facility repair",
            "broken equipment",
            "leak",
            "hvac",
            "heating",
            "cooling",
            "lighting",
            "plumbing",
            "cleaning request"
        ],
        "exclusions": [],
        "priority": 76,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "Facilities",
        "topic": "Office Info & Locations",
        "keywords": [
            "office address",
            "office location",
            "office hours",
            "office amenities",
            "cafeteria",
            "mail room",
            "directions to the office",
            "corporate address"
        ],
        "exclusions": [],
        "priority": 68,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "Facilities",
        "topic": "Health, Safety & Field",
        "keywords": [
            "health and safety",
            "workplace safety",
            "field safety",
            "safety guidance",
            "emergency action plan",
            "tick bite",
            "ticks",
            "mosquito",
            "repellent",
            "sunscreen",
            "uv protection",
            "thunderstorm",
            "ppe",
            "personal protective equipment",
            "workplace injury"
        ],
        "exclusions": [
            "ticket",
            "tickets"
        ],
        "priority": 92,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "Finance",
        "topic": "Project & Billing Codes",
        "keywords": [
            "billing code",
            "charge code",
            "project code",
            "phase and task",
            "rate code",
            "rate sheet",
            "project cost",
            "billing category"
        ],
        "exclusions": [],
        "priority": 78,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "Finance",
        "topic": "Expenses & Reimbursement",
        "keywords": [
            "expense",
            "expenses",
            "expense report",
            "reimbursement",
            "receipt",
            "mileage",
            "corporate card",
            "company credit card",
            "per diem"
        ],
        "exclusions": [],
        "priority": 86,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "Finance",
        "topic": "Procurement & Vendors",
        "keywords": [
            "procurement",
            "purchase order",
            "purchase orders",
            "po",
            "ariba",
            "vendor",
            "supplier",
            "requisition",
            "purchasing",
            "invoice approval"
        ],
        "exclusions": [],
        "priority": 88,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "Finance",
        "topic": "Travel",
        "keywords": [
            "travel booking",
            "travel policy",
            "business travel",
            "flight",
            "hotel booking",
            "rental car",
            "itinerary",
            "per diem",
            "visa"
        ],
        "exclusions": [],
        "priority": 76,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "Finance",
        "topic": "Accounting & Tax",
        "keywords": [
            "accounting",
            "tax form",
            "w-9",
            "w9",
            "invoice",
            "accounts payable",
            "accounts receivable",
            "certificate of insurance"
        ],
        "exclusions": [],
        "priority": 70,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "Legal",
        "topic": "Contracts & Agreements",
        "keywords": [
            "contract",
            "agreement",
            "nda",
            "confidentiality agreement",
            "clause",
            "change order",
            "proposal review",
            "signature request"
        ],
        "exclusions": [],
        "priority": 80,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "Legal",
        "topic": "Insurance & Certifications",
        "keywords": [
            "professional liability",
            "additional insured",
            "liability policy",
            "certificate of insurance",
            "state certification",
            "licensed to work"
        ],
        "exclusions": [],
        "priority": 74,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "Legal",
        "topic": "Compliance & Ethics",
        "keywords": [
            "anti-corruption",
            "ethics",
            "code of conduct",
            "compliance",
            "legal hold",
            "governance",
            "conflict of interest"
        ],
        "exclusions": [],
        "priority": 72,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "Knowledge",
        "topic": "Org Chart & Directory",
        "keywords": [
            "org chart",
            "organization chart",
            "reports to",
            "who reports",
            "direct report",
            "leadership team",
            "contact for",
            "who do i contact",
            "department leader",
            "business partner"
        ],
        "exclusions": [],
        "priority": 66,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "Knowledge",
        "topic": "Company Records & Data",
        "keywords": [
            "company records",
            "file storage",
            "legacy data",
            "legacy file",
            "where is the data",
            "upload documents",
            "document repository",
            "resume template",
            "project writeup"
        ],
        "exclusions": [],
        "priority": 64,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "Knowledge",
        "topic": "Definitions & Abbreviations",
        "keywords": [
            "stand for",
            "abbreviation",
            "definition",
            "what does this mean",
            "what is the meaning"
        ],
        "exclusions": [],
        "priority": 45,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "General",
        "topic": "Greeting & Smalltalk",
        "keywords": [
            "hello",
            "hi there",
            "good morning",
            "good afternoon",
            "good evening",
            "thank you",
            "thanks",
            "goodbye",
            "how are you"
        ],
        "exclusions": [],
        "priority": 10,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "*",
        "vertical": "General",
        "topic": "Agent Handoff & Escalation",
        "keywords": [
            "live agent",
            "human agent",
            "talk to someone",
            "representative",
            "escalate",
            "transfer me",
            "speak to a person"
        ],
        "exclusions": [],
        "priority": 70,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "es",
        "vertical": "IT",
        "topic": "IT Ticketing & Incidents",
        "keywords": [
            "ticket de soporte",
            "tickets de soporte",
            "incidente",
            "incidentes",
            "mesa de ayuda",
            "service desk",
            "mis tickets",
            "ticket abierto"
        ],
        "exclusions": [],
        "priority": 95,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "es",
        "vertical": "IT",
        "topic": "Accounts & Password",
        "keywords": [
            "contraseña",
            "restablecer contraseña",
            "reiniciar contraseña",
            "cuenta bloqueada",
            "autenticador",
            "iniciar sesión",
            "mfa"
        ],
        "exclusions": [],
        "priority": 90,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "es",
        "vertical": "IT",
        "topic": "Access & Permissions",
        "keywords": [
            "solicitar acceso",
            "conceder acceso",
            "permisos",
            "derechos de administrador",
            "grupo de seguridad"
        ],
        "exclusions": [],
        "priority": 86,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "es",
        "vertical": "IT",
        "topic": "Device & Hardware",
        "keywords": [
            "portátil",
            "computadora",
            "ordenador",
            "monitor",
            "impresora",
            "teclado",
            "ratón",
            "auriculares",
            "cámara web",
            "hardware"
        ],
        "exclusions": [],
        "priority": 80,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "es",
        "vertical": "IT",
        "topic": "Network & Connectivity",
        "keywords": [
            "vpn",
            "wifi",
            "red corporativa",
            "conexión a internet",
            "acceso remoto",
            "problema de red"
        ],
        "exclusions": [],
        "priority": 80,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "es",
        "vertical": "HR",
        "topic": "Benefits & Retirement",
        "keywords": [
            "beneficios",
            "seguro médico",
            "seguro dental",
            "plan médico",
            "pensión",
            "inscripción abierta"
        ],
        "exclusions": [],
        "priority": 90,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "es",
        "vertical": "HR",
        "topic": "Payroll & Compensation",
        "keywords": [
            "nómina",
            "salario",
            "pago",
            "cheque de pago",
            "depósito directo",
            "compensación",
            "bonificación"
        ],
        "exclusions": [],
        "priority": 88,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "es",
        "vertical": "HR",
        "topic": "Time Off & Attendance",
        "keywords": [
            "vacaciones",
            "días libres",
            "permiso",
            "baja por enfermedad",
            "tiempo libre",
            "asistencia",
            "hoja de horas",
            "licencia parental"
        ],
        "exclusions": [],
        "priority": 86,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "es",
        "vertical": "HR",
        "topic": "Policy & Handbook",
        "keywords": [
            "política de recursos humanos",
            "manual del empleado",
            "código de conducta",
            "política de trabajo remoto"
        ],
        "exclusions": [],
        "priority": 74,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "es",
        "vertical": "HR",
        "topic": "Onboarding & Offboarding",
        "keywords": [
            "incorporación",
            "nuevo empleado",
            "primer día",
            "orientación",
            "salida de empleado",
            "renuncia"
        ],
        "exclusions": [],
        "priority": 80,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "es",
        "vertical": "Facilities",
        "topic": "Building & Access",
        "keywords": [
            "acceso al edificio",
            "tarjeta de acceso",
            "credencial",
            "acceso a la oficina",
            "visitante"
        ],
        "exclusions": [],
        "priority": 84,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "es",
        "vertical": "Facilities",
        "topic": "Room & Desk Booking",
        "keywords": [
            "reservar una sala",
            "sala de reuniones",
            "sala de conferencias",
            "reservar escritorio",
            "espacio de trabajo"
        ],
        "exclusions": [],
        "priority": 80,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "es",
        "vertical": "Facilities",
        "topic": "Health, Safety & Field",
        "keywords": [
            "salud y seguridad",
            "seguridad laboral",
            "seguridad en campo",
            "equipo de protección personal",
            "epp",
            "plan de emergencia",
            "lesión laboral"
        ],
        "exclusions": [
            "ticket",
            "tickets"
        ],
        "priority": 92,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "es",
        "vertical": "Finance",
        "topic": "Expenses & Reimbursement",
        "keywords": [
            "gasto",
            "gastos",
            "informe de gastos",
            "reembolso",
            "recibo",
            "kilometraje",
            "tarjeta corporativa"
        ],
        "exclusions": [],
        "priority": 86,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "es",
        "vertical": "Finance",
        "topic": "Procurement & Vendors",
        "keywords": [
            "compras",
            "orden de compra",
            "órdenes de compra",
            "proveedor",
            "requisición",
            "ariba",
            "po"
        ],
        "exclusions": [],
        "priority": 88,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "es",
        "vertical": "Finance",
        "topic": "Travel",
        "keywords": [
            "viaje de negocios",
            "reserva de viaje",
            "vuelo",
            "hotel",
            "coche de alquiler",
            "itinerario",
            "visa"
        ],
        "exclusions": [],
        "priority": 76,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "es",
        "vertical": "Legal",
        "topic": "Contracts & Agreements",
        "keywords": [
            "contrato",
            "acuerdo",
            "acuerdo de confidencialidad",
            "cláusula",
            "firma"
        ],
        "exclusions": [],
        "priority": 80,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "es",
        "vertical": "General",
        "topic": "Greeting & Smalltalk",
        "keywords": [
            "hola",
            "buenos días",
            "buenas tardes",
            "buenas noches",
            "gracias",
            "adiós"
        ],
        "exclusions": [],
        "priority": 10,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "es",
        "vertical": "General",
        "topic": "Agent Handoff & Escalation",
        "keywords": [
            "agente humano",
            "hablar con una persona",
            "representante",
            "escalar",
            "transferir"
        ],
        "exclusions": [],
        "priority": 70,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "zh",
        "vertical": "IT",
        "topic": "IT Ticketing & Incidents",
        "keywords": [
            "工单",
            "支持工单",
            "服务工单",
            "事件",
            "帮助台",
            "服务台"
        ],
        "exclusions": [],
        "priority": 95,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "zh",
        "vertical": "IT",
        "topic": "Accounts & Password",
        "keywords": [
            "密码",
            "重置密码",
            "忘记密码",
            "账户锁定",
            "身份验证器",
            "登录",
            "多重身份验证"
        ],
        "exclusions": [],
        "priority": 90,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "zh",
        "vertical": "IT",
        "topic": "Access & Permissions",
        "keywords": [
            "申请访问权限",
            "授予访问权限",
            "权限",
            "管理员权限",
            "安全组"
        ],
        "exclusions": [],
        "priority": 86,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "zh",
        "vertical": "IT",
        "topic": "Device & Hardware",
        "keywords": [
            "笔记本电脑",
            "台式电脑",
            "显示器",
            "打印机",
            "键盘",
            "鼠标",
            "耳机",
            "摄像头",
            "硬件",
            "公司手机"
        ],
        "exclusions": [],
        "priority": 80,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "zh",
        "vertical": "IT",
        "topic": "Network & Connectivity",
        "keywords": [
            "虚拟专用网络",
            "无线网络",
            "公司网络",
            "互联网连接",
            "远程访问",
            "网络中断"
        ],
        "exclusions": [],
        "priority": 80,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "zh",
        "vertical": "HR",
        "topic": "Benefits & Retirement",
        "keywords": [
            "员工福利",
            "医疗保险",
            "牙科保险",
            "养老金",
            "开放注册",
            "保险范围"
        ],
        "exclusions": [],
        "priority": 90,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "zh",
        "vertical": "HR",
        "topic": "Payroll & Compensation",
        "keywords": [
            "工资单",
            "薪资",
            "发薪日",
            "工资",
            "直接存款",
            "奖金",
            "薪酬"
        ],
        "exclusions": [],
        "priority": 88,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "zh",
        "vertical": "HR",
        "topic": "Time Off & Attendance",
        "keywords": [
            "年假",
            "休假",
            "病假",
            "请假",
            "休息日",
            "考勤",
            "工时表",
            "育儿假"
        ],
        "exclusions": [],
        "priority": 86,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "zh",
        "vertical": "HR",
        "topic": "Policy & Handbook",
        "keywords": [
            "人力资源政策",
            "员工手册",
            "行为准则",
            "远程工作政策",
            "公司政策"
        ],
        "exclusions": [],
        "priority": 74,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "zh",
        "vertical": "HR",
        "topic": "Onboarding & Offboarding",
        "keywords": [
            "入职",
            "新员工",
            "第一天",
            "员工离职",
            "辞职",
            "离职流程"
        ],
        "exclusions": [],
        "priority": 80,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "zh",
        "vertical": "Facilities",
        "topic": "Building & Access",
        "keywords": [
            "大楼门禁",
            "门禁卡",
            "工牌",
            "办公室门禁",
            "访客通行"
        ],
        "exclusions": [],
        "priority": 84,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "zh",
        "vertical": "Facilities",
        "topic": "Room & Desk Booking",
        "keywords": [
            "预订会议室",
            "会议室",
            "预订工位",
            "共享工位",
            "工作区预订"
        ],
        "exclusions": [],
        "priority": 80,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "zh",
        "vertical": "Facilities",
        "topic": "Health, Safety & Field",
        "keywords": [
            "健康与安全",
            "工作场所安全",
            "现场安全",
            "个人防护装备",
            "安全指南",
            "应急行动计划",
            "工伤"
        ],
        "exclusions": [],
        "priority": 92,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "zh",
        "vertical": "Finance",
        "topic": "Expenses & Reimbursement",
        "keywords": [
            "费用",
            "报销",
            "费用报销",
            "收据",
            "里程",
            "公司卡",
            "差旅补贴"
        ],
        "exclusions": [],
        "priority": 86,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "zh",
        "vertical": "Finance",
        "topic": "Procurement & Vendors",
        "keywords": [
            "采购",
            "采购订单",
            "供应商",
            "请购单",
            "订单审批",
            "ariba",
            "po"
        ],
        "exclusions": [],
        "priority": 88,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "zh",
        "vertical": "Finance",
        "topic": "Travel",
        "keywords": [
            "商务旅行",
            "差旅预订",
            "航班",
            "酒店预订",
            "租车",
            "行程",
            "签证"
        ],
        "exclusions": [],
        "priority": 76,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "zh",
        "vertical": "Legal",
        "topic": "Contracts & Agreements",
        "keywords": [
            "合同",
            "协议",
            "保密协议",
            "条款",
            "签名"
        ],
        "exclusions": [],
        "priority": 80,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "zh",
        "vertical": "General",
        "topic": "Greeting & Smalltalk",
        "keywords": [
            "你好",
            "早上好",
            "下午好",
            "晚上好",
            "谢谢",
            "再见"
        ],
        "exclusions": [],
        "priority": 10,
        "rule_source": "BuiltIn"
    },
    {
        "locale": "zh",
        "vertical": "General",
        "topic": "Agent Handoff & Escalation",
        "keywords": [
            "人工客服",
            "转人工",
            "与真人交谈",
            "代表",
            "升级处理"
        ],
        "exclusions": [],
        "priority": 70,
        "rule_source": "BuiltIn"
    }
]
_TOPIC_SEPARATORS = re.compile(r'''[\s.,;:!?()\[\]{}<>/\\\-_+=*&^%$#@~`'"]+''')

def _load_customer_topic_rules(path):
    if not path:
        return []
    from pathlib import Path as _Path
    _path = _Path(path)
    if not _path.exists():
        print(f'Customer topic overrides not found at {path}; using built-in taxonomy only.')
        return []
    _frame = pd.read_csv(_path, keep_default_na=False)
    _required = {'Enabled', 'Locale', 'Vertical', 'CanonicalTopic', 'Keywords', 'Exclusions', 'Priority'}
    _missing = sorted(_required - set(_frame.columns))
    if _missing:
        raise ValueError(f'Customer topic overrides are missing columns: {_missing}')
    _enabled = _frame[_frame['Enabled'].astype(str).str.strip().str.lower().isin({'1', 'true', 'yes', 'y'})]
    _rules = []
    for _row_number, _row in _enabled.iterrows():
        _vertical = str(_row['Vertical']).strip()
        _topic = str(_row['CanonicalTopic']).strip()
        _keywords = [value.strip() for value in str(_row['Keywords']).split('|') if value.strip()]
        if not _vertical or not _topic or not _keywords:
            raise ValueError(f'Enabled customer topic rule at row {_row_number + 2} is incomplete.')
        _rules.append({
            'locale': str(_row['Locale']).strip().lower() or '*',
            'vertical': _vertical,
            'topic': _topic,
            'keywords': _keywords,
            'exclusions': [value.strip() for value in str(_row['Exclusions']).split('|') if value.strip()],
            'priority': int(_row['Priority']),
            'rule_source': 'Customer',
        })
    print(f'Loaded {len(_rules):,} approved customer topic override rule(s).')
    return _rules

CUSTOMER_TOPIC_RULES = _load_customer_topic_rules(TOPIC_OVERRIDES_FILE)
TOPIC_RULES = CUSTOMER_TOPIC_RULES + TOPIC_TAXONOMY

def _normalize_topic_text(value):
    return ' '.join(part for part in _TOPIC_SEPARATORS.split(str(value or '').lower()) if part)

def _topic_language(locale):
    return str(locale or '').strip().lower()[:2]

def _contains_topic_term(normalized_text, term, language):
    normalized_term = _normalize_topic_text(term)
    if not normalized_term:
        return False
    if language == 'zh':
        return normalized_term in normalized_text
    return f' {normalized_term} ' in f' {normalized_text} '

def _topic_term_weight(term, language):
    normalized_term = _normalize_topic_text(term)
    if language == 'zh':
        return 3 if len(normalized_term) >= 4 else 2
    if len(normalized_term.split()) >= 2:
        return 3
    return 1 if len(normalized_term) <= 2 else 2

def classify_topic_keyword(text, locale, taxonomy):
    normalized_text = _normalize_topic_text(text)
    language = _topic_language(locale)
    if not normalized_text:
        return {
            'topic': 'No User Intent',
            'vertical': 'System-Init',
            'matched_terms': None,
            'score': 0.0,
            'confidence': 1.0,
            'ambiguous': False,
            'language': language,
            'source': 'no-user-intent',
            'rule_source': '',
            'version': TOPIC_CLASSIFIER_VERSION,
        }

    scored = []
    for rule in taxonomy:
        if rule['locale'] not in ('*', language):
            continue
        hits = [term for term in rule['keywords']
                if _contains_topic_term(normalized_text, term, language)]
        excluded = any(_contains_topic_term(normalized_text, term, language)
                       for term in rule['exclusions'])
        score = 0 if excluded else sum(_topic_term_weight(term, language) for term in hits)
        if score:
            scored.append((score, rule['priority'], rule, hits))

    if not scored:
        return {
            'topic': 'Other / Uncategorized',
            'vertical': 'Unknown',
            'matched_terms': None,
            'score': 0.0,
            'confidence': 0.0,
            'ambiguous': False,
            'language': language,
            'source': 'uncategorized',
            'rule_source': '',
            'version': TOPIC_CLASSIFIER_VERSION,
        }

    scored.sort(key=lambda item: (-item[0], -item[1], item[2]['topic']))
    top_score, top_priority = scored[0][0], scored[0][1]
    tied = [item for item in scored if item[0] == top_score and item[1] == top_priority]
    tied_topics = {(item[2]['vertical'], item[2]['topic']) for item in tied}
    ambiguous = len(tied_topics) > 1
    confidence = (0.4 if ambiguous else
                  0.98 if top_score >= 6 else
                  0.93 if top_score >= 4 else
                  0.85 if top_score >= 2 else 0.70)

    if ambiguous:
        return {
            'topic': 'Other / Uncategorized',
            'vertical': 'Unknown',
            'matched_terms': ' | '.join(sorted({term for item in tied for term in item[3]})),
            'score': float(top_score),
            'confidence': confidence,
            'ambiguous': True,
            'language': language,
            'source': 'uncategorized',
            'rule_source': '',
            'version': TOPIC_CLASSIFIER_VERSION,
        }

    _, _, rule, hits = scored[0]
    rule_source = rule.get('rule_source', 'BuiltIn')
    return {
        'topic': rule['topic'],
        'vertical': rule['vertical'],
        'matched_terms': ' | '.join(hits),
        'score': float(top_score),
        'confidence': confidence,
        'ambiguous': False,
        'language': language,
        'source': 'keyword',
        'rule_source': rule_source,
        'version': TOPIC_CLASSIFIER_VERSION + ('+CUSTOMER' if rule_source == 'Customer' else ''),
    }

def classify_topic_row(row):
    native_topic = str(row.get('primary_topic_native') or '').strip()
    if native_topic and native_topic not in ('No Topic Detected', 'No User Intent', 'Other / Uncategorized'):
        return {
            'topic': native_topic,
            'vertical': 'Native',
            'matched_terms': None,
            'score': None,
            'confidence': 1.0,
            'ambiguous': False,
            'language': _topic_language(row.get('locale')),
            'source': 'native',
            'rule_source': 'Native',
            'version': 'Native',
        }
    return classify_topic_keyword(row.get('first_user_prompt'), row.get('locale'), TOPIC_RULES)

_classified = sessions.apply(classify_topic_row, axis=1)
sessions['primary_topic_derived'] = [item['topic'] for item in _classified]
sessions['topic_vertical_derived'] = [item['vertical'] for item in _classified]
sessions['topic_classification_source'] = [item['source'] for item in _classified]
sessions['topic_matched_terms'] = [item['matched_terms'] for item in _classified]
sessions['topic_match_score'] = [item['score'] for item in _classified]
sessions['topic_confidence'] = [item['confidence'] for item in _classified]
sessions['topic_ambiguous'] = [item['ambiguous'] for item in _classified]
sessions['topic_language'] = [item['language'] for item in _classified]
sessions['topic_rule_source'] = [item['rule_source'] for item in _classified]
sessions['topic_classifier_version'] = [item['version'] for item in _classified]

_source_counts = sessions['topic_classification_source'].value_counts()
print(f"Stage 1 classification: {int(_source_counts.get('keyword', 0)):,} matched / "
      f"{int(_source_counts.get('native', 0)):,} native / {len(sessions):,} total; "
      f"{int(_source_counts.get('uncategorized', 0)):,} uncategorized")


In [ ]:
# === 7c-2. LOCAL MULTILINGUAL DISCOVERY FALLBACK (offline) ===
ENABLE_SEMANTIC_FALLBACK = True
MIN_UNCATEGORIZED_FOR_CLUSTERING = 20
N_TERMS_PER_CLUSTER_LABEL = 3

_uncategorized_mask = sessions['topic_classification_source'] == 'uncategorized'
_uncategorized_count = int(_uncategorized_mask.sum())

if not ENABLE_SEMANTIC_FALLBACK:
    print(f'Stage 2 disabled -- {_uncategorized_count:,} row(s) remain uncategorized.')
elif _uncategorized_count < MIN_UNCATEGORIZED_FOR_CLUSTERING:
    print(f'Stage 2 skipped -- only {_uncategorized_count:,} uncategorized row(s).')
else:
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.cluster import KMeans
    except ImportError as _import_err:
        print(f'Stage 2 skipped -- scikit-learn unavailable ({_import_err}).')
    else:
        _texts = sessions.loc[_uncategorized_mask, 'first_user_prompt'].fillna('')
        _nonblank_mask = _texts.str.strip() != ''
        _cluster_texts = _texts[_nonblank_mask]
        if len(_cluster_texts) < MIN_UNCATEGORIZED_FOR_CLUSTERING:
            print(f'Stage 2 skipped -- only {len(_cluster_texts):,} nonblank rows.')
        else:
            _n_clusters = max(2, min(10, len(_cluster_texts) // 10))
            try:
                # Character n-grams work for English, whitespace languages, and CJK without
                # downloading a model or sending prompts outside the workspace.
                _vectorizer = TfidfVectorizer(
                    analyzer='char_wb',
                    ngram_range=(2, 5),
                    max_df=0.95,
                    min_df=2,
                    lowercase=True)
                _matrix = _vectorizer.fit_transform(_cluster_texts)
                _kmeans = KMeans(n_clusters=_n_clusters, random_state=42, n_init=10)
                _cluster_ids = _kmeans.fit_predict(_matrix)
            except ValueError as _cluster_err:
                print(f'Stage 2 skipped -- clustering failed ({_cluster_err}).')
            else:
                _terms = _vectorizer.get_feature_names_out()
                _cluster_labels = {}
                _cluster_terms = {}
                for _cid in range(_n_clusters):
                    _center = _kmeans.cluster_centers_[_cid]
                    _top_idx = _center.argsort()[::-1][:N_TERMS_PER_CLUSTER_LABEL]
                    _top_terms = [_terms[i].strip() for i in _top_idx if _center[i] > 0 and _terms[i].strip()]
                    _cluster_terms[_cid] = _top_terms
                    _cluster_labels[_cid] = ('Auto-Discovered: ' + ', '.join(_top_terms)) if _top_terms else 'Auto-Discovered: (unlabeled cluster)'

                for _index, _cluster_id in zip(_cluster_texts.index, _cluster_ids):
                    sessions.at[_index, 'primary_topic_derived'] = _cluster_labels[_cluster_id]
                    sessions.at[_index, 'topic_classification_source'] = 'auto-discovered'
                    sessions.at[_index, 'topic_matched_terms'] = ' | '.join(_cluster_terms[_cluster_id])
                    sessions.at[_index, 'topic_match_score'] = 0.0
                    sessions.at[_index, 'topic_confidence'] = 0.50
                    sessions.at[_index, 'topic_ambiguous'] = False
                    sessions.at[_index, 'topic_rule_source'] = 'AutoDiscovered'
                    sessions.at[_index, 'topic_classifier_version'] = TOPIC_CLASSIFIER_VERSION + '+TFIDF'
                print(f'Stage 2: {len(_cluster_texts):,} row(s) clustered into {_n_clusters} candidate topic(s).')

_final_counts = sessions['topic_classification_source'].value_counts()
print('\nTopic classification summary:')
for _source in ('keyword', 'auto-discovered', 'uncategorized', 'no-user-intent'):
    print(f'  {_source:16} {int(_final_counts.get(_source, 0)):>7,}')


## 8b. Agent Performance KPI fact — `agent_performance`

Per-conversation KPI table that powers the **Agent Performance / Agent Evaluation** pages. Every column is derived directly from the raw transcript activities, so it **replaces the old `conversation_kpis_flattened.csv`** — no separate analytics extract is needed. `SessionType`, `SessionOutcome`, `OutcomeReason` and `ImpliedSuccess` are documented heuristics; the model's calculated columns (`PromptQualityScore`, `ConversationType`, `ConversationComplexity`, `PromptLength`, `BotFriendlyName`) are computed downstream in DAX and intentionally not produced here.

In [ ]:
def build_agent_performance(parsed):
    rows = []
    for _, r in parsed.iterrows():
        acts = get_activities(r['content_json'])
        msgs = [a for a in acts if isinstance(a, dict) and a.get('type') == 'message']
        user_msgs = [a for a in msgs if (a.get('from') or {}).get('role') in (1, 'user')]
        bot_msgs  = [a for a in msgs if (a.get('from') or {}).get('role') in (0, 'bot')]
        # Robust agent identity (same resolver as build_sessions): keeps BotName keyed
        # consistently with sessions.primary_agent_schema and populates a real BotId.
        primary_agent, _agent_name, _agent_bot_id, connected_agents = resolve_agent_identity(acts, r)

        topics, last_intent_id = [], None
        for a in acts:
            if isinstance(a, dict) and a.get('valueType') == 'IntentRecognition':
                v = a.get('value') or {}
                t = v.get('intentTitle')
                if t and t not in topics:
                    topics.append(t)
                last_intent_id = v.get('intentId') or last_intent_id

        sys_error = user_error = False
        for a in acts:
            if isinstance(a, dict) and a.get('valueType') == 'ErrorTraceData':
                v = a.get('value') or {}
                if v.get('isUserError'):
                    user_error = True
                else:
                    sys_error = True

        latency_total = plan_steps = trace_count = event_count = 0
        for a in acts:
            if not isinstance(a, dict):
                continue
            if a.get('type') == 'trace':
                trace_count += 1
            elif a.get('type') == 'event':
                event_count += 1
            v = a.get('value') if isinstance(a.get('value'), dict) else None
            if v and 'executionTime' in v:
                plan_steps += 1
                m = re.match(r'(\d+):(\d+):(\d+\.?\d*)', str(v.get('executionTime')))
                if m:
                    h, mn, s = int(m.group(1)), int(m.group(2)), float(m.group(3))
                    latency_total += int((h*3600 + mn*60 + s) * 1000)
        avg_latency = int(latency_total / plan_steps) if plan_steps else 0

        first_ts = last_ts = None
        for a in acts:
            ts = a.get('timestampMs') if isinstance(a, dict) else None
            if ts:
                first_ts = ts if first_ts is None or ts < first_ts else first_ts
                last_ts  = ts if last_ts  is None or ts > last_ts  else last_ts
        duration_s = int((last_ts - first_ts) / 1000) if first_ts and last_ts else 0

        def _txt(a):
            return (a.get('text') or '')[:TEXT_TRUNCATE] if isinstance(a, dict) else ''
        first_user = _txt(user_msgs[0]) if user_msgs else ''
        last_user  = _txt(user_msgs[-1]) if user_msgs else ''
        first_bot  = _txt(bot_msgs[0]) if bot_msgs else ''
        last_bot   = _txt(bot_msgs[-1]) if bot_msgs else ''

        upn = None
        for a in acts:
            if not isinstance(a, dict):
                continue
            v = a.get('value') if isinstance(a.get('value'), dict) else {}
            vars_ = v.get('variables') if isinstance(v.get('variables'), dict) else {}
            upn = upn or vars_.get('ESS_UserContext_UPN') or vars_.get('Var_ESS_UserContext_UPN')

        # --- Native outcome first; heuristics only when SessionInfo is absent. ---
        session_info_values = [
            a.get('value') or {}
            for a in acts
            if isinstance(a, dict)
            and (a.get('valueType') == 'SessionInfo' or a.get('name') == 'SessionInfo')
            and isinstance(a.get('value'), dict)]
        session_info = session_info_values[-1] if session_info_values else {}
        explicit_outcome_raw = str(
            session_info.get('outcome')
            or session_info.get('lastSessionOutcome')
            or '').strip()
        explicit_outcome = {
            'resolved': 'Resolved',
            'escalated': 'Escalated',
            'abandon': 'Abandoned',
            'abandoned': 'Abandoned',
        }.get(explicit_outcome_raw.lower(), explicit_outcome_raw or None)
        explicit_reason = (
            session_info.get('outcomeReason')
            or session_info.get('lastSessionOutcomeReason'))
        engaged = bool(topics) or len(bot_msgs) > 0
        if explicit_outcome:
            outcome = explicit_outcome
            session_type = str(session_info.get('type') or ('Engaged' if engaged else 'Unengaged'))
        else:
            session_type = 'Engaged' if engaged else 'Unengaged'
            if connected_agents:
                outcome = 'Escalated'
            elif not engaged or (len(user_msgs) > 0 and len(bot_msgs) == 0) or (sys_error and len(bot_msgs) == 0):
                outcome = 'Abandoned'
            else:
                outcome = 'Engaged-Unclassified'
        outcome_reason = explicit_reason or (
            'SystemError' if sys_error else 'UserError' if user_error else
            ('Escalated' if outcome == 'Escalated' else
             'Abandoned' if outcome == 'Abandoned' else 'NoExplicitOutcome'))
        explicit_success = session_info.get('impliedSuccess')
        if explicit_success is None:
            implied_success = 'TRUE' if (outcome == 'Resolved' and not sys_error) else 'FALSE'
        else:
            implied_success = 'TRUE' if bool(explicit_success) else 'FALSE'

        start_iso = _iso(first_ts) or r.get('conversationstarttime', '')
        end_iso = _iso(last_ts) or ''

        # --- contract columns to match the agent_performance Delta schema (M parser parity) ---
        _perf_errcodes = [ (a.get('value') or {}).get('errorCode')
                           for a in acts if isinstance(a, dict) and a.get('valueType') == 'ErrorTraceData' ]
        _perf_first_err = next((e for e in _perf_errcodes if e), None)
        _perf_authed = any(((a.get('from') or {}).get('aadObjectId')) for a in user_msgs)
        _perf_connected = set()
        for _pa in acts:
            if isinstance(_pa, dict):
                _pv = _pa.get('value') or {}
                if isinstance(_pv, dict) and _pv.get('connectedAgentBotSchemaName'):
                    _perf_connected.add(_pv.get('connectedAgentBotSchemaName'))
        _perf_gen = sum(1 for a in acts if isinstance(a, dict)
                        and a.get('valueType') in ('GenerativeAIResponseTraceData', 'GenerativeResponseTraceData'))
        rows.append({
            **_identity_fields(r),
            'ConversationTranscriptId': r.get('conversationtranscriptid', ''),
            'ConversationStartTime': r.get('conversationstarttime', '') or start_iso,
            'SchemaVersion': r.get('schemaversion', ''),
            'BotName': primary_agent or '',
            'BotId': _agent_bot_id or r.get('botid', ''),
            'AADTenantId': r.get('aadtenantid', ''),
            'BatchId': r.get('batchid', ''),
            'SessionStartTimeUtc': start_iso,
            'SessionEndTimeUtc': end_iso,
            'SessionType': session_type,
            'SessionOutcome': outcome,
            'TurnCount': len(msgs),
            'OutcomeReason': outcome_reason,
            'ImpliedSuccess': implied_success,
            'LastUserIntentId': last_intent_id,
            'IsDesignMode': 'FALSE',
            'Locale': r.get('locale', ''),
            'UserMessageCount': len(user_msgs),
            'BotMessageCount': len(bot_msgs),
            'TotalMessageCount': len(msgs),
            'FirstUserMessage': first_user,
            'LastUserMessage': last_user,
            'FirstBotMessage': first_bot,
            'LastBotMessage': last_bot,
            'TopicsTriggered': '|'.join(topics) if topics else '',
            'TopicCount': len(topics),
            'PrimaryTopic': topics[0] if topics else '',
            'DurationSeconds': duration_s,
            'AverageLatencyMs': avg_latency,
            'Var_ESS_UserContext_UPN': upn,
            'Var_ESS_Message_Disclaimer': '',
            'Var_ESS_UserContext_RefreshInterval_Hours': '',
            'Var_ESS_UserContext_Conversation_Initialized': '',
            'AllVariables': '',
            'TotalActivityCount': len(acts),
            'TraceActivityCount': trace_count,
            'EventActivityCount': event_count,
            'StatusCode': 1 if outcome == 'Resolved' else 0,
            'StateCode': 1,
            'FirstErrorCode': _perf_first_err,
            'ErrorCategory': None,
            'IsAuthenticated': 'TRUE' if _perf_authed else 'FALSE',
            'MultiAgentSession': 'TRUE' if _perf_connected else 'FALSE',
            'SessionOutcomeExplicit': explicit_outcome,
            'SessionOutcomeReasonExplicit': explicit_reason,
            'GenerativeResponseCount': _perf_gen,
            'LLMCallCount': None,
            'PromptTokenCount': None,
            'CompletionTokenCount': None,
            'TotalTokenCount': None,
            'PluginCallCount': None,
            'PluginSuccessCount': None,
            'TopicsStarted': len(topics),
            'TopicsCompleted': None,
            'TopicCompletionRate': None,
            'PrimaryTopicDwellMs': None,
            'TotalTopicDwellMs': None,
        })
    return pd.DataFrame(rows, columns=_IDENTITY_COLUMNS + ['ConversationTranscriptId', 'ConversationStartTime', 'SchemaVersion', 'BotName', 'BotId', 'AADTenantId', 'BatchId', 'SessionStartTimeUtc', 'SessionEndTimeUtc', 'SessionType', 'SessionOutcome', 'TurnCount', 'OutcomeReason', 'ImpliedSuccess', 'LastUserIntentId', 'IsDesignMode', 'Locale', 'UserMessageCount', 'BotMessageCount', 'TotalMessageCount', 'FirstUserMessage', 'LastUserMessage', 'FirstBotMessage', 'LastBotMessage', 'TopicsTriggered', 'TopicCount', 'PrimaryTopic', 'DurationSeconds', 'AverageLatencyMs', 'Var_ESS_UserContext_UPN', 'Var_ESS_Message_Disclaimer', 'Var_ESS_UserContext_RefreshInterval_Hours', 'Var_ESS_UserContext_Conversation_Initialized', 'AllVariables', 'TotalActivityCount', 'TraceActivityCount', 'EventActivityCount', 'StatusCode', 'StateCode', 'FirstErrorCode', 'ErrorCategory', 'IsAuthenticated', 'MultiAgentSession', 'SessionOutcomeExplicit', 'SessionOutcomeReasonExplicit', 'GenerativeResponseCount', 'LLMCallCount', 'PromptTokenCount', 'CompletionTokenCount', 'TotalTokenCount', 'PluginCallCount', 'PluginSuccessCount', 'TopicsStarted', 'TopicsCompleted', 'TopicCompletionRate', 'PrimaryTopicDwellMs', 'TotalTopicDwellMs'])

agent_performance = build_agent_performance(parsed)
if not child_summary.empty:
    _performance_child = child_summary.rename(columns={
        'parent_agent_name': 'ParentAgentName',
        'parent_agent_id': 'ParentAgentId',
        'child_agent_names': 'ChildAgentNames',
        'child_agent_schema_names': 'ChildAgentSchemaNames',
        'child_agent_ids': 'ChildAgentIds',
        'child_agent_invocation_states': 'ChildAgentInvocationStates',
        'child_agent_dialogs': 'ChildAgentDialogs',
        'child_agent_plan_steps': 'ChildAgentPlanSteps',
        'child_agent_invocation_count': 'ChildAgentInvocationCount',
        'child_agent_count': 'ChildAgentCount',
        'has_child_agent': 'HasChildAgent',
        'child_agent_detail': 'ChildAgentDetail',
    })
    agent_performance = agent_performance.merge(
        _performance_child, on='conversation_natural_key', how='left',
        validate='one_to_one')
_performance_child_defaults = {
    'ParentAgentName': None,
    'ParentAgentId': None,
    'ChildAgentNames': None,
    'ChildAgentSchemaNames': None,
    'ChildAgentIds': None,
    'ChildAgentInvocationStates': None,
    'ChildAgentDialogs': None,
    'ChildAgentPlanSteps': None,
    'ChildAgentInvocationCount': 0,
    'ChildAgentCount': 0,
    'HasChildAgent': False,
    'ChildAgentDetail': None,
}
for _column, _default in _performance_child_defaults.items():
    if _column not in agent_performance.columns:
        agent_performance[_column] = _default
    else:
        agent_performance[_column] = agent_performance[_column].where(
            pd.notna(agent_performance[_column]), _default)
print(f'agent performance rows: {len(agent_performance):,}')

## 9. Write Delta tables

First carries `SourceEnvironment` through to the transcript-derived tables (`agent_sessions`, `agent_turns`, `agent_errors`, `agent_subagents`, `agent_performance` get a `source_environment` column via a conversation→environment lookup; `agent_catalogue` gets `source_environments`, the pipe-joined list of environments each agent appears in). Then flattens embedded newlines/tabs in every text cell (one record per physical line — matches the pandas + Power Query behaviour), converts each pandas frame to Spark with an explicit all-string schema, and writes the six Delta tables. The supplied Fabric V2 template imports these tables through the Lakehouse SQL analytics endpoint. Direct Lake would require a separately authored semantic model and isn't provided by this repository.


In [ ]:
# --- Unified feedback events: transcript reactions + registry-mapped external exports ---
_USER_FEEDBACK_COLUMNS = [
    'Feedback Id', 'Comment', 'Translated Comment', 'Comment Language', 'Date Submitted UTC',
    'Feedback Type', 'Microsoft Response Status', 'App', 'App Language', 'Platform', 'Source Type',
    'Logs, Attachments', 'User Id', 'User Email', 'Browser', 'Browser Version', 'AI Context Prompt',
    'AI Context Response Message', 'Survey Question', 'Survey Response Option', 'Additional Metadata',
    'Date Submitted Date', 'Sentiment', 'Feedback Event Key', 'Conversation Id', 'Agent Id', 'Channel',
    'Feedback Source', 'Feedback Verdict', 'Match Status', 'canonical_agent_key',
    'canonical_agent_name', 'canonical_environment_key', 'agent_mapping_status',
    'raw_agent_id', 'raw_agent_schema', 'raw_agent_name', 'source_environment'
]
_FEEDBACK_QUARANTINE_COLUMNS = [
    'Feedback Event Key', 'Feedback Id', 'Conversation Id', 'Agent Id', 'Channel',
    'Feedback Source', 'Match Status', 'canonical_agent_key', 'canonical_agent_name',
    'canonical_environment_key', 'agent_mapping_status', 'raw_agent_id',
    'raw_agent_schema', 'raw_agent_name'
]

def _feedback_metadata(conversation_id, agent_id, agent_name, channel, existing=''):
    if existing:
        return existing
    return json.dumps({
        'conversationId': conversation_id or '',
        'essAgentId': agent_id or '',
        'aiAgents': [{'Name': agent_name or '', 'Type': 'Custom'}],
        'copilotType': 'm365' if channel == 'Microsoft 365 Copilot Chat' else 'custom',
        'UiHost': channel or '',
    }, ensure_ascii=False)

def _registry_key_match(value, registry_rows, active_keys):
    normalized = str(value or '').strip().casefold()
    matches = [
        row for row in registry_rows
        if row['Enabled']
        and row['AgentKey'] in active_keys
        and row['AgentKey'].casefold() == normalized]
    if not matches:
        return None
    result = dict(matches[0])
    environment_values = [
        str(row.get('EnvironmentKey') or '').strip() for row in matches]
    environments = set(environment_values)
    result['EnvironmentKey'] = (
        next(iter(environments))
        if all(environment_values) and len(environments) == 1
        else '')
    return result

def resolve_external_feedback_agent(
        item, session_frame, registry_rows, active_keys):
    raw_agent_id = str(item.get('AgentId') or '').strip()
    raw_agent_name = str(item.get('AgentName') or '').strip()
    match = (
        _registry_key_match(raw_agent_id, registry_rows, active_keys)
        or resolve_registry_alias(
            registry_rows, 'M365Title', raw_agent_id, active_keys))
    matched_session = None
    conversation_id = str(item.get('ConversationId') or '').strip()
    if conversation_id:
        candidates = session_frame[
            session_frame['conversation_id'].astype(str) == conversation_id]
        candidates = candidates[
            candidates['agent_mapping_status'].astype(str) == 'RegistryMapped']
        if match is not None:
            candidates = candidates[
                candidates['canonical_agent_key'].astype(str) == match['AgentKey']]
            registry_environment = str(match.get('EnvironmentKey') or '').strip()
            if registry_environment:
                candidates = candidates[
                    candidates['canonical_environment_key'].astype(str)
                    == registry_environment]
        if len(candidates) == 1:
            matched_session = candidates.iloc[0]
            if match is None:
                match = _registry_key_match(
                    matched_session.get('canonical_agent_key'),
                    registry_rows, active_keys)
    if match is None:
        return None, matched_session
    environment = match['EnvironmentKey']
    if matched_session is not None:
        environment = (
            str(matched_session.get('canonical_environment_key') or '')
            or environment)
    return {
        'canonical_agent_key': match['AgentKey'],
        'canonical_agent_name': match['AgentLabel'],
        'canonical_environment_key': environment,
        'agent_mapping_status': 'RegistryMapped',
        'raw_agent_id': raw_agent_id,
        'raw_agent_schema': '',
        'raw_agent_name': raw_agent_name,
        'source_environment': (
            str(matched_session.get('source_environment') or '')
            if matched_session is not None else ''),
    }, matched_session

def build_user_feedback(session_frame, external_path):
    rows, quarantine = [], []

    def _text(value):
        return '' if value is None or pd.isna(value) else str(value)

    from pathlib import Path as _Path
    if external_path and _Path(external_path).exists():
        external = pd.read_csv(external_path, keep_default_na=False)
        required = {
            'FeedbackEventId', 'ConversationId', 'AgentId', 'AgentName', 'Channel',
            'FeedbackSource', 'FeedbackVerdict', 'FeedbackComment', 'SubmittedUtc',
            'UserId', 'UserEmail', 'Prompt', 'Response', 'App', 'AppLanguage',
            'Platform', 'SourceType', 'AdditionalMetadata', 'MatchStatus'}
        missing = sorted(required - set(external.columns))
        if missing:
            raise ValueError(f'Feedback events file is missing columns: {missing}')
        for _, item in external.iterrows():
            conversation_id = _text(item.get('ConversationId')).strip()
            event_id = (
                _text(item.get('FeedbackEventId')).strip()
                or f'{conversation_id}|{_text(item.get("FeedbackVerdict"))}')
            event_key = f'External|{event_id}'
            identity, matched = resolve_external_feedback_agent(
                item, session_frame, AGENT_REGISTRY_ROWS, ACTIVE_AGENT_KEYS)
            if identity is None:
                quarantine.append({
                    'Feedback Event Key': event_key,
                    'Feedback Id': event_id,
                    'Conversation Id': conversation_id,
                    'Agent Id': _text(item.get('AgentId')),
                    'Channel': _text(item.get('Channel')) or 'Unknown',
                    'Feedback Source': _text(item.get('FeedbackSource')),
                    'Match Status': 'UnmappedRegistryAgent',
                    'canonical_agent_key': '',
                    'canonical_agent_name': '',
                    'canonical_environment_key': '',
                    'agent_mapping_status': 'UnmappedQuarantined',
                    'raw_agent_id': _text(item.get('AgentId')),
                    'raw_agent_schema': '',
                    'raw_agent_name': _text(item.get('AgentName')),
                })
                continue
            agent_id = identity['canonical_agent_key']
            agent_name = identity['canonical_agent_name']
            channel = _text(item.get('Channel')) or 'Unknown'
            metadata = _feedback_metadata(
                conversation_id, agent_id, agent_name, channel,
                _text(item.get('AdditionalMetadata')))
            row = {
                'Feedback Id': event_id,
                'Comment': _text(item.get('FeedbackComment')),
                'Translated Comment': '',
                'Comment Language': '',
                'Date Submitted UTC': _text(item.get('SubmittedUtc')),
                'Feedback Type': _text(item.get('FeedbackVerdict')),
                'Microsoft Response Status': 'Completed',
                'App': _text(item.get('App')) or agent_name,
                'App Language': _text(item.get('AppLanguage')),
                'Platform': _text(item.get('Platform')) or channel,
                'Source Type': _text(item.get('SourceType')),
                'Logs, Attachments': '',
                'User Id': _text(item.get('UserId')),
                'User Email': _text(item.get('UserEmail')),
                'Browser': '',
                'Browser Version': '',
                'AI Context Prompt': _text(item.get('Prompt')),
                'AI Context Response Message': _text(item.get('Response')),
                'Survey Question': '',
                'Survey Response Option': '',
                'Additional Metadata': metadata,
                'Date Submitted Date': _text(item.get('SubmittedUtc'))[:10],
                'Sentiment': _text(item.get('FeedbackVerdict')),
                'Feedback Event Key': event_key,
                'Conversation Id': conversation_id,
                'Agent Id': agent_id,
                'Channel': channel,
                'Feedback Source': _text(item.get('FeedbackSource')),
                'Feedback Verdict': _text(item.get('FeedbackVerdict')),
                'Match Status': (
                    'Exact conversation match' if matched is not None
                    else 'Registry agent match'),
                '_conversation_natural_key': (
                    _text(matched.get('conversation_natural_key'))
                    if matched is not None else ''),
                '_source_rank': 0,
            }
            row.update(identity)
            rows.append(row)
    elif external_path:
        print(f'External feedback events not found at {external_path}; using transcript reactions only.')

    for _, item in session_frame.iterrows():
        verdict = _text(item.get('feedback_verdict')).strip()
        if not verdict:
            continue
        display_verdict = (
            'Thumbs Up' if verdict == 'Positive'
            else 'Thumbs Down' if verdict == 'Negative' else verdict)
        conversation_id = _text(item.get('conversation_id'))
        agent_id = _text(item.get('canonical_agent_key'))
        agent_name = _text(item.get('canonical_agent_name'))
        submitted = _text(item.get('session_start_utc'))
        row = {
            'Feedback Id': conversation_id,
            'Comment': _text(item.get('feedback_comment')),
            'Translated Comment': '',
            'Comment Language': '',
            'Date Submitted UTC': submitted,
            'Feedback Type': display_verdict,
            'Microsoft Response Status': 'Completed',
            'App': agent_name,
            'App Language': _text(item.get('locale')),
            'Platform': 'Copilot Studio',
            'Source Type': 'Conversation Transcript',
            'Logs, Attachments': '',
            'User Id': _text(item.get('user_id_hash')),
            'User Email': '',
            'Browser': '',
            'Browser Version': '',
            'AI Context Prompt': _text(item.get('first_user_prompt')),
            'AI Context Response Message': '',
            'Survey Question': '',
            'Survey Response Option': '',
            'Additional Metadata': _feedback_metadata(
                conversation_id, agent_id, agent_name, 'Copilot Studio'),
            'Date Submitted Date': submitted[:10],
            'Sentiment': (
                'Positive' if display_verdict == 'Thumbs Up'
                else 'Negative' if display_verdict == 'Thumbs Down'
                else display_verdict),
            'Feedback Event Key': (
                f'Transcript|{item.get("conversation_natural_key")}|'
                f'{display_verdict.lower()}'),
            'Conversation Id': conversation_id,
            'Agent Id': agent_id,
            'Channel': 'Copilot Studio',
            'Feedback Source': 'Conversation Transcript',
            'Feedback Verdict': display_verdict,
            'Match Status': 'Native transcript',
            'canonical_agent_key': agent_id,
            'canonical_agent_name': agent_name,
            'canonical_environment_key': _text(item.get('canonical_environment_key')),
            'agent_mapping_status': _text(item.get('agent_mapping_status')),
            'raw_agent_id': _text(item.get('raw_agent_id')),
            'raw_agent_schema': _text(item.get('raw_agent_schema')),
            'raw_agent_name': _text(item.get('raw_agent_name')),
            'source_environment': _text(item.get('source_environment')),
            '_conversation_natural_key': _text(
                item.get('conversation_natural_key')),
            '_source_rank': 1,
        }
        rows.append(row)

    if rows:
        frame = pd.DataFrame(rows)
        external_rows = (
            frame[frame['_source_rank'] == 0]
            .drop_duplicates('Feedback Event Key', keep='first'))
        transcript_rows = frame[frame['_source_rank'] == 1].copy()
        external_pairs = {
            (str(row['_conversation_natural_key']).casefold(),
             str(row['canonical_agent_key']).casefold(),
             str(row['Feedback Verdict']).casefold())
            for _, row in external_rows.iterrows()
            if str(row['Match Status']) == 'Exact conversation match'
            and str(row['_conversation_natural_key'] or '')}
        transcript_rows = transcript_rows[
            ~transcript_rows.apply(
                lambda row: (
                    str(row['_conversation_natural_key']).casefold(),
                    str(row['canonical_agent_key']).casefold(),
                    str(row['Feedback Verdict']).casefold()) in external_pairs,
                axis=1)]
        user_feedback = pd.concat(
            [external_rows, transcript_rows], ignore_index=True)
        user_feedback = user_feedback[_USER_FEEDBACK_COLUMNS].reset_index(drop=True)
    else:
        user_feedback = pd.DataFrame(columns=_USER_FEEDBACK_COLUMNS)
    feedback_quarantine = (
        pd.DataFrame(quarantine, columns=_FEEDBACK_QUARANTINE_COLUMNS)
        .drop_duplicates('Feedback Event Key', keep='first')
        .reset_index(drop=True))
    return user_feedback, feedback_quarantine

user_feedback, user_feedback_quarantine = build_user_feedback(
    sessions, FEEDBACK_EVENTS_FILE)
print(
    f'unified feedback events: {len(user_feedback):,}; '
    f'quarantined identifiers: {len(user_feedback_quarantine):,}')

outputs = {
    'agent_sessions':             sessions,
    'agent_turns':                turns,
    'agent_errors':               errors,
    'agent_subagents':            subagents,
    'agent_catalogue':            agent_dim,
    'agent_performance':          agent_performance,
    'user_feedback':              user_feedback,
    'user_feedback_quarantine':   user_feedback_quarantine,
}

_ws = re.compile(r'[\r\n\t]+')
def _flatten(v):
    return _ws.sub(' ', v).strip() if isinstance(v, str) else v

from pyspark.sql.types import StructType, StructField, StringType

# --- Upsert support (WRITE_MODE='merge'): keyed MERGE so re-running an incremental
# load over the SAME environments updates rows in place instead of duplicating them.
MERGE_KEYS = {
    'agent_sessions':             ['conversation_natural_key'],
    'agent_turns':                ['conversation_natural_key', 'turn_id'],
    'agent_errors':               ['conversation_natural_key', 'error_timestamp_utc', 'error_code'],
    'agent_subagents':            ['conversation_natural_key', 'subagent_event_key'],
    'agent_catalogue':            ['canonical_agent_key'],
    'agent_performance':          ['conversation_natural_key'],
    'user_feedback':              ['canonical_environment_key', 'Feedback Event Key'],
    'user_feedback_quarantine':   ['Feedback Event Key'],
}

def _merge_delta(sdf, name, table):
    keys = list(MERGE_KEYS.get(name, []))
    from functools import reduce
    from delta.tables import DeltaTable
    from pyspark.sql import functions as F
    missing_keys = [k for k in keys if k not in sdf.columns]
    if missing_keys:
        raise ValueError(f'{name}: merge key column(s) missing: {missing_keys}')
    if keys:
        null_key = reduce(lambda a, b: a | b, [F.col(k).isNull() for k in keys])
        if sdf.filter(null_key).limit(1).count():
            raise ValueError(f'{name}: merge key contains null values: {keys}')
        if sdf.groupBy(*keys).count().filter(F.col('count') > 1).limit(1).count():
            raise ValueError(f'{name}: merge key is not unique in the incoming batch: {keys}')
    if not keys or not spark.catalog.tableExists(table):
        # First load (or no natural key) -> create/extend via append; enables column mapping.
        (sdf.write.mode('append')
            .option('mergeSchema', 'true')
            .option('delta.columnMapping.mode', 'name')
            .option('delta.minReaderVersion', '2')
            .option('delta.minWriterVersion', '5')
            .format('delta').saveAsTable(table))
        return 'append(first-load)'
    cond = ' AND '.join([f't.`{k}` <=> s.`{k}`' for k in keys])
    (DeltaTable.forName(spark, table).alias('t')
        .merge(sdf.alias('s'), cond)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    return f'merge(on {"+".join(keys)})'


def _validate_full_incoming(pdf, name):
    keys = list(MERGE_KEYS.get(name, []))
    missing = [key for key in keys if key not in pdf.columns]
    if missing:
        raise ValueError(f'{name}: merge key column(s) missing: {missing}')
    if not keys or pdf.empty:
        return
    null_or_blank = pd.Series(False, index=pdf.index)
    for key in keys:
        null_or_blank = null_or_blank | pdf[key].isna() | (pdf[key].astype(str).str.strip() == '')
    if null_or_blank.any():
        raise ValueError(f'{name}: merge key contains null/blank values: {keys}')
    if pdf.duplicated(keys, keep=False).any():
        raise ValueError(
            f'{name}: merge key is not unique in the full incoming batch: {keys}')

_chunk_rows = int(globals().get('PANDAS_TO_SPARK_CHUNK_ROWS', 20000))

for name, pdf in outputs.items():
    pdf = pdf.copy()
    _validate_full_incoming(pdf, name)
    for col in pdf.columns:
        pdf[col] = pdf[col].map(_flatten)
    # All-string write keeps the schema stable across snapshots; the PBIP types on read.
    pdf = pdf.astype(object).where(pd.notna(pdf), None)
    # Force an explicit all-string schema so that columns which are entirely null in a
    # given snapshot (e.g. feedback_verdict when no feedback was submitted) are written as
    # StringType, NOT Spark's void/NullType — otherwise the Lakehouse SQL endpoint won't
    # expose them and the PBIP errors with 'the column ... wasn't found'.
    schema = StructType([StructField(c, StringType(), True) for c in pdf.columns])
    table = f'{OUTPUT_PREFIX}.{name}' if OUTPUT_PREFIX else name
    _pdf_str = pdf.astype(str).where(pd.notna(pdf), None)

    # Write in chunks to avoid hitting spark.rpc.message.maxSize on large tenants.
    if WRITE_MODE == 'merge':
        _how = 'merge(no-rows)'
        _chunks = 0
        if len(_pdf_str) == 0 and not spark.catalog.tableExists(table):
            (spark.createDataFrame([], schema=schema)
                .write.mode('append')
                .option('mergeSchema', 'true')
                .option('delta.columnMapping.mode', 'name')
                .option('delta.minReaderVersion', '2')
                .option('delta.minWriterVersion', '5')
                .format('delta').saveAsTable(table))
            _how = 'append(empty-first-load)'
            _chunks = 1
        else:
            for _i in range(0, len(_pdf_str), _chunk_rows):
                _part = _pdf_str.iloc[_i:_i + _chunk_rows]
                _sdf_part = spark.createDataFrame(_part, schema=schema)
                _how = _merge_delta(_sdf_part, name, table)
                _chunks += 1
        print(f'  {table:28} {len(pdf):>7,} rows  ->  {_how}, chunks={_chunks}, chunk_rows={_chunk_rows}')
    else:
        if WRITE_MODE == 'overwrite':
            spark.sql(f'DROP TABLE IF EXISTS {table}')
        _chunks = 0
        if len(_pdf_str) == 0:
            (spark.createDataFrame([], schema=schema)
                .write.mode('append')
                .option('overwriteSchema', 'true')
                .option('delta.columnMapping.mode', 'name')
                .option('delta.minReaderVersion', '2')
                .option('delta.minWriterVersion', '5')
                .format('delta').saveAsTable(table))
            _chunks = 1
        else:
            for _i in range(0, len(_pdf_str), _chunk_rows):
                _part = _pdf_str.iloc[_i:_i + _chunk_rows]
                _mode = 'append' if (WRITE_MODE == 'overwrite' or _chunks > 0) else WRITE_MODE
                (spark.createDataFrame(_part, schema=schema)
                    .write.mode(_mode)
                    .option('overwriteSchema', 'true')
                    .option('delta.columnMapping.mode', 'name')
                    .option('delta.minReaderVersion', '2')
                    .option('delta.minWriterVersion', '5')
                    .format('delta').saveAsTable(table))
                _chunks += 1
        print(f'  {table:28} {len(pdf):>7,} rows  ->  written ({WRITE_MODE}), chunks={_chunks}, chunk_rows={_chunk_rows}')

print('\nDone. Eight Delta tables written. Refresh the supplied Fabric V2 Import model to pick them up.')
